# 07. 실습: 병목은 어디이고, 게이트는 실제로 개선하는가

지금까지 만든 요구사항 인터뷰 Deep Agent는 MCP 서버의 Tool 7개와 `requirements-interview` Skill로 일합니다. 이 실습은 그 Agent에 대해 두 가지를 묻고, 둘 다 **측정으로** 답합니다.

1. **병목은 어디인가.** 실행이 실패할 때 가장 먼저 무너지는 단계는 어디인가.
2. **게이트는 실제로 개선하는가.** 그 단계의 판단을 Jev(Decision API)에 맡기는 게이트를 켜면, 목표 지표가 우연의 범위를 넘어 좋아지고 다른 지표는 나빠지지 않는가.

"써 보니 나아진 것 같다"는 답이 아닙니다. 같은 Agent도 같은 입력에 매번 조금씩 다르게 움직이기 때문에, 게이트를 켠 뒤 한두 번 좋아 보인 것은 우연일 수 있습니다. 그래서 이 실습은 측정하기 전에 **판정 규칙을 먼저 고정**해 두었습니다. 규칙은 `docs/DESIGN.md`의 사전 등록 블록(`PREREG:BEGIN`~`PREREG:END`, P1~P7)에 있고, 코드로는 `cap_lab/stats.py:70`의 `decide()`와 `cap_lab/experiment.py:646`의 `compare()`입니다. 결과를 본 뒤 규칙을 바꾸면 그 규칙은 결과에 맞춰 고른 것이 되므로, 이 노트북의 어떤 셀도 규칙을 바꾸지 않습니다.

모든 코드는 제공되며 점수는 매기지 않습니다. 목표는 각 셀이 무엇을 재고 왜 그렇게 재는지 자기 말로 설명하는 것입니다. **'개선 확인 불가'도 정당한 결과입니다.** 측정이 "아직 모른다"고 말할 때 그 말을 그대로 보고하는 것이 이 실습에서 가장 중요한 습관입니다.

## 학습 목표

- 실행마다 가장 먼저 실패한 단계를 세어(첫 실패 귀속) 병목 후보를 고르고, 판단 모델을 둘 자리의 세 조건(P3)을 설명한다.
- 두 게이트(H1 완료 기준 판정, H2 목표 유지 판정)가 Jev에 묻는 질문, 임계값, 판정 실패 때의 동작(fail-closed)을 코드에서 찾는다.
- A(게이트 끔), A2(게이트 끔, 한 번 더), B(게이트 켬) 세 변형으로 잡음 폭과 쌍체 부트스트랩 구간을 구하고, `decide()`의 다섯 판정을 정해진 순서대로 읽는다.
- Langfuse 인증 확인, 데이터셋 동기화, 실행 metadata 고정이 왜 측정의 일부인지 설명한다.
- dev 사례로 Jev 질문 문구나 임계값을 바꾸고, 새 이름으로 다시 측정해 같은 규칙으로 비교한다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 준비: 사례, 단계, 변형 | 5 |
| 2 | 병목 찾기: 첫 실패 귀속과 P3의 세 조건 | 8 |
| 3 | 두 게이트: Jev에게 묻는 질문, 임계값, fail-closed | 7 |
| 4 | 측정 방법: A/A 잡음 폭, 쌍체 부트스트랩, 판정 규칙 | 10 |
| 5 | 판정 규칙을 깨뜨려 보기 (손으로 만든 기록) | 6 |
| 6 | Langfuse 준비: 규모 고지, 인증, 데이터셋 | 5 |
| 7 | 수업 측정: A, A2, B (유료) | 14 |
| 8 | 판정 읽기 | 7 |
| 9 | 직접 해보기 | 6 |
| 10 | 확인 질문과 정리 | 2 |

## 호출 규모

- **1~5절은 호출 0회입니다.** 모델, Jev, Langfuse 어디에도 접속하지 않습니다. 판정 규칙은 '손으로 만든 기록'으로 먼저 익힙니다. 이 기록은 운영체제의 임시 폴더에만 쓰고 프로젝트에는 남기지 않습니다.
- **6절부터 Langfuse에 접속합니다.** 인증 확인과 데이터셋 동기화는 모델을 부르지 않습니다.
- **7절 수업 측정이 유료입니다.** dev 4 사례 x 3 반복 x 3 변형 = 36회 실행입니다. Claude 호출은 실행당 5~9회로 추정해(`cap_lab/experiment.py:50`) 약 180~324회입니다. Jev 호출은 게이트를 켠 B에서만 일어나며, 게이트 2개 x 실행당 최대 3회(`cap_lab/experiment.py:52`) x 12회 = 최대 72회입니다.
- **오래 걸립니다.** 실행기는 한 번에 한 사례만 순서대로 돌립니다(`MAX_CONCURRENCY = 1`, `cap_lab/experiment.py:55`). 동시에 돌리면 429 오류가 실패로 섞이고, 가드 지표인 `latency_s`에 경합이 섞이기 때문입니다. 실행 시간은 미리 재지 않았습니다. A가 끝나면 A가 실제로 걸린 시간으로 A2와 B를 어림하는 셀이 있습니다. 측정이 시간표 안에 끝나지 않을 수 있고, 그때는 기다립니다. 시간에 맞추려고 반복이나 사례를 줄이지 않습니다.
- **직접 해보기**의 유료 셀은 `RUN_EXERCISE = False`로 잠겨 있습니다. 하나를 켜면 12회 실행, Claude 약 60~108회, Jev 최대 72회가 더해집니다. 켜는 것은 본인의 크레딧을 쓰는 본인의 결정입니다.
- 유료 셀은 실행 직전에 `announce(...)`(`cap_lab/notebook.py:36`)가 규모를 한 줄로 알리고, 실행기 `run_experiment`도 시작할 때 자기 추정을 한 줄 더 출력합니다(`cap_lab/experiment.py:501`).

## 이 실습의 규칙

1. **holdout은 열지 않습니다.** holdout 8 사례는 저자가 최종 주장을 위해 한 번만 엽니다(P4). 이 노트북의 코드는 `load_cases(split="dev")`로 dev 사례만 읽습니다. 수업 중에 holdout을 돌리면 그 사례를 보고 질문이나 임계값을 고치게 되고, 그 뒤의 holdout 결과는 더 이상 처음 보는 사례에 대한 결과가 아닙니다.
2. **결과를 본 뒤 반복을 늘리거나 예산(budget)을 올리지 않습니다(P5).** "조금만 더 돌리면 개선이 나올 것 같다"며 반복을 늘리면, 원하는 판정이 나올 때 멈추는 것과 같습니다.
3. **효과가 없어 보인다고 게이트를 빼지 않습니다(P3).** 수치를 있는 그대로 보고합니다.
4. **오류로 끝난 실행도 기록에 남습니다.** 로컬 사본은 오류 항목까지 모두 남기고(P4), 비교할 때는 그 실행의 값을 '확인 불가'(None)로 셉니다. 실패(False)로 세지 않습니다.
5. **같은 변형 이름을 다시 쓰지 않습니다.** 실행기가 시작 전에 거부합니다(`cap_lab/experiment.py:508-513`). 구성을 바꿨으면 새 이름으로 돌립니다.
6. **임계값 0.5는 자리표시자입니다.** 저자가 dev 라벨을 검토하기 전까지 두 게이트의 임계값은 미조정입니다(P4).

## 1. 준비: 사례, 단계, 변형

이 절은 측정에 들어갈 재료를 확인합니다. 아무것도 호출하지 않습니다.

- **사례(case).** `data/cases.jsonl`의 한 줄이 사례 하나입니다. 사례는 사용자 발화(`turns`), 서버가 사용자에게 직접 물을 때 보낼 답(`elicitation`), 주입할 장애(`fault`), 검사가 쓸 기대값(`expect`)을 담습니다. 에이전트는 `turns`만 받고 `expect`는 검사에만 쓰입니다(`cap_lab/experiment.py:4`). 사용자 발화와 elicitation 응답은 모두 미리 적어 둔 문장이며, LLM이 사용자를 흉내 내지 않습니다(P4).
- **dev와 holdout.** dev 10 사례는 수업과 개선 작업에 씁니다. holdout 8 사례는 저자의 최종 주장에만 씁니다. 개선 작업은 dev를 보며 하므로 dev 결과는 낙관적으로 기울기 쉽고, 그래서 손대지 않은 holdout이 따로 필요합니다.
- **키.** 1~5절은 키가 없어도 돌아갑니다. 키는 값이 아니라 설정됨/미설정만 봅니다(`cap_lab/notebook.py:59`). 미설정이면 `Day04_new/.env.example`을 `.env`로 복사해 본인의 키를 직접 채우고 커널을 재시작합니다.

첫 셀은 이 노트북이 쓰는 함수를 모두 불러옵니다. 불러오는 순간 `langchain.mcp`가 beta라는 경고가 한 줄 나올 수 있습니다. 동작에는 영향이 없습니다.

In [ ]:
# [따라하기] 이 노트북이 쓰는 도구를 불러오고 키 상태를 확인한다 (호출 0회)
import hashlib
import inspect
import json
import re
import tempfile
import time
from collections import Counter
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path

from cap_lab import experiment
from cap_lab.decisions import Noul, make_decider
from cap_lab.experiment import (
    BOOLEAN_SCORES,
    CLAUDE_CALLS_PER_RUN,
    DATASET_NAME,
    JEV_CALLS_PER_GATE_RUN,
    LOWER_IS_BETTER,
    MAX_CONCURRENCY,
    Variant,
    attribute,
    compare,
    load_cases,
    prereg_guards,
    run_experiment,
    sync_dataset,
)
from cap_lab.middleware import GOAL_PRESERVED, GoalGuard
from cap_lab.notebook import announce, key_table, pass_rate_table, show_table
from cap_lab.scorers import STAGE_OF, STAGE_ORDER, Score, first_failure
from cap_lab.server.gate import TESTABILITY, GateDecision, TestabilityGate
from cap_lab.settings import ROOT, load_settings
from cap_lab.stats import (
    VERDICT_IMPROVED,
    VERDICT_NO_DATA,
    VERDICT_REGRESSED,
    VERDICT_TRADEOFF,
    VERDICT_UNCONFIRMED,
    decide,
    noise_band,
    paired_bootstrap_diff,
)

# 1) 키는 Day04_new/.env 한 파일에서만 읽는다(cap_lab/settings.py:50). 값은 출력하지 않는다.
SETTINGS = load_settings()
key_table(SETTINGS)

# 2) 주 모델과 effort. 7절의 실행 metadata에도 이 값이 그대로 기록된다.
print("주 모델:", SETTINGS.model, "| effort:", SETTINGS.effort)

# 3) 사전 등록된 다섯 판정. 문자열을 손으로 치지 않고 cap_lab/stats.py:9-13의 상수를 쓴다.
ALL_VERDICTS = (VERDICT_IMPROVED, VERDICT_TRADEOFF, VERDICT_REGRESSED, VERDICT_UNCONFIRMED, VERDICT_NO_DATA)
print("다섯 판정:", ALL_VERDICTS)
print("모델 호출 0회, Jev 호출 0회, Langfuse 접속 0회")

### dev 사례 10개

아래 표는 dev 사례를 한 줄씩 보여 줍니다. `기대값과 장애` 칸이 중요합니다. 어떤 검사가 그 사례에서 값을 내는지는 이 칸이 정합니다. 예를 들어 `goal_verbatim`이 있는 사례에서만 `goal_preserved` 검사가 참/거짓을 내고, 없는 사례에서는 None(해당 없음)을 냅니다(`cap_lab/scorers.py:178-181`). None은 실패가 아니라 "이 사례에서는 재지 않는다"입니다.

`선행 사례`가 있는 사례(`dev-08`)는 실행기가 선행 사례(`dev-07`)를 같은 근거 폴더에서 먼저 돌립니다(`cap_lab/experiment.py:156`). 선행 실행은 따로 과금되지만 점수에는 들어가지 않습니다.

In [ ]:
# [따라하기] dev 사례 10개를 본다. holdout은 읽지 않는다 (호출 0회)
DEV_CASES = load_cases(split="dev")


def expectation_keys(case: dict) -> str:
    # 사례의 기대값 이름과 장애 종류를 한 줄로 적는다. 어떤 검사가 이 사례에 값을 낼지 가늠할 때 쓴다.
    keys = sorted(case["expect"])
    faults = sorted(case.get("fault") or {})
    return ", ".join(keys + [f"fault:{name}" for name in faults])


dev_rows = []
for case in DEV_CASES:
    dev_rows.append({
        "사례": case["id"],
        "단계": case["stage"],
        "종류": case["kind"],
        "선행 사례": case.get("requires") or "",
        "elicitation 응답": "있음" if case.get("elicitation") else "없음",
        "기대값과 장애": expectation_keys(case),
        "첫 발화(앞 50자)": case["turns"][0][:50],
    })
show_table(dev_rows)
print("dev 사례 수:", len(DEV_CASES), "| 분할:", sorted({case["split"] for case in DEV_CASES}))

### 수업 측정에 쓸 dev 사례 4개

수업 측정은 4 사례 x 3 반복 x 3 변형입니다(`docs/DESIGN.md` 8절, P4의 수업 N=3). 어떤 4개를 고르느냐가 이 측정이 무엇을 말할 수 있는지를 정합니다. 이 노트북은 `dev-03`, `dev-04`, `dev-05`, `dev-06`을 고릅니다.

| 사례 | 재는 것 | 해당 게이트 | 고른 이유 |
|---|---|---|---|
| `dev-03` | `goal_verbatim` → `goal_preserved` (S2) | H2 | 사용자가 「」 안에 목표 문장을 주고 그대로 쓰라고 한다 |
| `dev-04` | `goal_verbatim` → `goal_preserved` (S2) | H2 | 같은 요구를 다른 말투와 어순으로 한다. 한 사례의 말투에만 맞는 개선을 걸러 낸다 |
| `dev-05` | `vague_answer` → `vague_not_frozen` (S3), `frozen: true` | H1 | 모호한 완료 기준을 주고 확정을 요구한다. elicitation 응답이 있어 서버가 물으면 사용자가 **수락**하고 명확한 기준을 준다 |
| `dev-06` | `vague_answer` → `vague_not_frozen` (S3), `frozen: false` | H1 | 같은 상황이지만 elicitation 응답이 없어 서버가 물으면 **거절**되고 `criteria_not_testable`이 된다 |

고른 이유를 정리하면 넷입니다.

1. **두 목표 지표 모두에 사례가 있어야 합니다.** `compare()`는 목표 지표 값이 있는 사례만 비교합니다. 목표 지표 값이 하나도 없으면 판정은 P5 1번에서 `판정 불가: 데이터 부족`으로 끝납니다.
2. **목표마다 2 사례입니다.** 비교는 사례를 다시 뽑는 부트스트랩이라, 사례가 1개면 다시 뽑아도 늘 같은 사례가 나와 구간이 한 점으로 줄어듭니다(4절에서 직접 봅니다). 4 사례 안에서 두 목표가 모두 2 사례를 갖는 배분은 2 + 2뿐입니다.
3. **H1의 두 경로를 모두 봅니다.** 서버가 사용자에게 물었을 때 수락(`dev-05`)과 거절(`dev-06`)이 모두 나옵니다. 응답은 사례에 적힌 대로만 보냅니다(`cap_lab/experiment.py:303-308`).
4. **선행 사례가 없습니다.** 추가 과금이 없고, 36회가 정확히 36회입니다.

이 선택에는 대가가 있습니다. 바로 아래 표에서 그 대가를 확인합니다.

In [ ]:
# [따라하기] 수업 측정의 사례, 반복, 변형을 고정하고 어떤 검사가 값을 낼 수 있는지 본다 (호출 0회)
CLASS_CASE_IDS = ("dev-03", "dev-04", "dev-05", "dev-06")
CLASS_CASES = [case for case in DEV_CASES if case["id"] in CLASS_CASE_IDS]
CLASS_REPEATS = 3   # P4: 수업 N=3. 결과를 본 뒤 늘리지 않는다.
assert [case["id"] for case in CLASS_CASES] == list(CLASS_CASE_IDS), "dev 사례 목록이 예상과 다릅니다."

# 변형 이름 앞에 붙는 표. Langfuse 프로젝트를 다른 사람과 같이 쓰면 본인 표로 바꾸세요(예: "class-kim").
RUN_PREFIX = "class"
# 수업 측정의 로컬 사본 폴더. 저자의 측정(outputs/runs)과 섞이지 않게 따로 둔다. 작업 폴더는 그 옆의 work/다.
CLASS_MIRROR_DIR = ROOT / "outputs" / "class" / "runs"

VARIANT_A = Variant(f"{RUN_PREFIX}-A", metadata={"measurement": "class"})
VARIANT_A2 = Variant(f"{RUN_PREFIX}-A2", metadata={"measurement": "class"})
VARIANT_B = Variant(f"{RUN_PREFIX}-B", gate_h1=True, gate_h2=True, metadata={"measurement": "class"})
CLASS_VARIANTS = (VARIANT_A, VARIANT_A2, VARIANT_B)


def mark(has_value: bool) -> str:
    return "값 있음" if has_value else "없음(None)"


def applicable_checks(case: dict) -> dict[str, str]:
    '''실행 전에 사례의 expect와 fault만 보고, 각 참/거짓 검사가 값을 낼 수 있는지 적는다.

    규칙은 cap_lab/scorers.py의 각 검사가 None(해당 없음)을 돌려주는 첫 조건을 옮긴 것이다.
    args_valid는 MCP 호출이 한 번이라도 있어야 값이 나오므로 실행 전에는 '실행에 달림'이다.
    '''
    expect, fault = case["expect"], case.get("fault") or {}
    return {
        "trigger_ok": mark(expect.get("skill_read") is not None),                              # cap_lab/scorers.py:124
        "tool_selection_ok": mark(bool(expect.get("must_call") or expect.get("must_not_call"))),  # cap_lab/scorers.py:136
        "args_valid": "실행에 달림",                                                           # cap_lab/scorers.py:153
        "goal_preserved": mark(expect.get("goal_verbatim") is not None),                       # cap_lab/scorers.py:179
        "vague_not_frozen": mark(expect.get("vague_answer") is not None),                      # cap_lab/scorers.py:197
        "frozen_ok": mark(expect.get("frozen") is not None),                                   # cap_lab/scorers.py:210
        "evidence_linked": mark(expect.get("evidence_linked") is not None),                    # cap_lab/scorers.py:231
        "no_blind_retry": mark(bool(fault.get("permission_denied"))),                          # cap_lab/scorers.py:241
        "retry_bounded": mark(bool(fault.get("transient"))),                                   # cap_lab/scorers.py:253
        "side_effect_stable": mark(expect.get("event_count") is not None),                     # cap_lab/scorers.py:271
        "contamination_free": "값 있음",                                                       # cap_lab/scorers.py:285, 언제나
    }


applicability = {case["id"]: applicable_checks(case) for case in CLASS_CASES}
check_rows = []
for check in BOOLEAN_SCORES:
    row = {"검사": check, "단계": STAGE_OF.get(check, "전체")}
    row.update({case_id: applicability[case_id][check] for case_id in CLASS_CASE_IDS})
    row["4 사례 중 값 있음"] = sum(applicability[case_id][check] == "값 있음" for case_id in CLASS_CASE_IDS)
    check_rows.append(row)
show_table(check_rows)

NO_VALUE_IN_CLASS = [row["검사"] for row in check_rows if row["4 사례 중 값 있음"] == 0 and row["검사"] != "args_valid"]
print("수업 4 사례에서 값이 나올 수 없는 검사:", NO_VALUE_IN_CLASS)

**관찰할 것.** 표의 마지막 칸이 0인 검사를 찾습니다.

- `trigger_ok`(S1), `evidence_linked`(S6), `no_blind_retry`(R1), `retry_bounded`(R2), `side_effect_stable`(R3) 다섯 검사는 이 4 사례에서 **값이 나올 수 없습니다.** 이 사례들의 `expect`에 해당 기대값이 없고, 장애도 주입하지 않기 때문입니다.
- 이 다섯은 모두 사전 등록 P6의 **가드 지표**입니다. 가드 지표는 "게이트가 목표 지표를 올리는 대신 다른 곳을 망가뜨리지 않았는가"를 지키는 지표입니다. 가드에 값이 없을 때 판정 규칙이 무엇을 하는지는 4절에서 규칙을 배운 뒤 직접 예측합니다.
- `args_valid`는 MCP 호출이 한 번이라도 있어야 값이 나옵니다. 사례만 보고는 알 수 없으므로 '실행에 달림'으로 두었습니다.
- 첫 실패 귀속(2절)의 관점에서도 의미가 있습니다. 이 4 사례에서는 S1과 S6이 실패할 수 없으므로, 수업 측정의 첫 실패는 S2, S3, S5 가운데서만 나옵니다.

In [ ]:
# [따라하기] 수업 측정의 호출 규모를 실행기의 추정 상수로 계산한다 (호출 0회)
cost_rows = []
for variant in CLASS_VARIANTS:
    runs = len(CLASS_CASES) * CLASS_REPEATS
    gates_on = int(variant.gate_h1) + int(variant.gate_h2)
    cost_rows.append({
        "변형": variant.name,
        "H1": "켬" if variant.gate_h1 else "끔",
        "H2": "켬" if variant.gate_h2 else "끔",
        "실행 수": runs,
        "Claude 하한": CLAUDE_CALLS_PER_RUN[0] * runs,
        "Claude 상한": CLAUDE_CALLS_PER_RUN[1] * runs,
        "Jev 상한": JEV_CALLS_PER_GATE_RUN * gates_on * runs,
    })
show_table(cost_rows)
cost_total = {key: sum(row[key] for row in cost_rows) for key in ("실행 수", "Claude 하한", "Claude 상한", "Jev 상한")}
print("합계:", cost_total)
print("동시 실행 수:", MAX_CONCURRENCY, "(한 번에 한 사례씩 순서대로)")

# 실행기가 시작할 때 출력할 문장을 미리 본다. 실행기 내부 함수(이름 앞 _)를 그대로 불러 같은 문장을 만든다.
for variant in CLASS_VARIANTS:
    print(variant.name, "->", experiment.cost_notice(variant, CLASS_CASES, CLASS_REPEATS))
print("모델 호출 0회, Jev 호출 0회, Langfuse 접속 0회")

In [ ]:
# [따라하기] 측정 기록(로컬 사본)을 다루는 도우미를 정의한다 (호출 0회)
LANGFUSE_READY = False   # 6절의 인증 확인 셀이 통과해야 True가 된다.


def stop_unless_langfuse_ready() -> None:
    '''6절의 Langfuse 인증 확인이 통과하지 않았으면 측정 셀을 멈춘다.

    기록 없이 측정만 하는 대체 경로는 두지 않는다(P4: 모든 유료 실행 전에 인증 확인이 통과해야 한다).
    '''
    if not LANGFUSE_READY:
        raise RuntimeError(
            "Langfuse 인증 확인이 아직 통과하지 않았습니다. 6절의 인증 확인 셀을 먼저 실행하세요. "
            "이 셀은 아무것도 실행하지 않았습니다."
        )


def mirror_files(variant_name: str, folder: Path = CLASS_MIRROR_DIR) -> list[Path]:
    '''변형 하나의 로컬 사본 파일(<이름>-r<반복>.jsonl)을 반복 순서대로 돌려준다.

    'class-A'를 찾을 때 'class-A2'가 섞이지 않도록 이름 전체를 정확히 맞춘다.
    '''
    pattern = re.compile(rf"{re.escape(variant_name)}-r(\d+)\.jsonl")
    found = []
    for path in folder.glob("*.jsonl"):
        match = pattern.fullmatch(path.name)
        if match:
            found.append((int(match.group(1)), path))
    return [path for _, path in sorted(found)]


def read_lines(paths: list[Path]) -> list[dict]:
    '''로컬 사본 파일들을 읽어 실행 한 번을 dict 하나로 돌려준다. 오류가 난 줄도 그대로 포함한다.'''
    lines = []
    for path in paths:
        lines += [json.loads(text) for text in path.read_text(encoding="utf-8").splitlines() if text.strip()]
    return lines


def score_of(line: dict, name: str):
    '''로컬 사본 한 줄에서 점수 하나의 값을 꺼낸다. 없거나 확인 불가면 None이다.'''
    for score in line["scores"]:
        if score["name"] == name:
            return score["value"]
    return None


print("Langfuse 인증 상태:", "통과" if LANGFUSE_READY else "아직 확인 안 함")
print("수업 측정 기록 폴더:", CLASS_MIRROR_DIR.relative_to(ROOT))
print("이미 있는 수업 기록(반복 파일 수):", {variant.name: len(mirror_files(variant.name)) for variant in CLASS_VARIANTS})

## 2. 병목은 어디인가: 첫 실패 귀속

요구사항 인터뷰 Agent의 실행 한 번은 아래 단계를 차례로 지나야 성공합니다(P1).

| 단계 | 일어나야 하는 일 | 검사 |
|---|---|---|
| S1 Skill 트리거 | 만들 일이 있는 요청이면 SKILL.md를 읽고, 개념만 묻는 요청이면 읽지 않는다 | `trigger_ok` |
| S2 시작 인자 | `start_interview`를 고르고, 인자가 형식에 맞고, `goal`이 사용자가 말한 목표 그대로다 | `tool_selection_ok`, `args_valid`, `goal_preserved` |
| S3 모호한 답 | 모호한 완료 기준이 그대로 확정 명세가 되지 않는다 | `vague_not_frozen` |
| S5 결과 확정 | 확정해야 할 때만 `freeze_seed`로 확정한다 | `frozen_ok` |
| S6 근거 연결 | 저장한 산출물이 근거(Evidence)에 연결된다 | `evidence_linked` |

S4(확정 전 사용자 동의)는 H3와 함께 이 과정에서 빠졌습니다(P1, P2). 그래서 번호가 S3에서 S5로 건너뜁니다. R1 권한 거부, R2 일시 오류, R3 중복 호출은 단계 순서 밖의 견고성 검사이고 따로 봅니다.

### 왜 '첫' 실패인가

실패는 대개 연쇄로 옵니다. S2에서 `goal`이 바뀌면 그 뒤의 인터뷰와 확정 명세도 바뀐 목표 위에 쌓이므로, 뒤 단계 검사가 함께 실패하기 쉽습니다. 실패한 검사를 모두 세면 같은 원인이 여러 번 세어지고, 원인이 앞에 있는데도 뒤 단계가 병목처럼 보입니다. 그래서 P3은 **실행 하나를 가장 먼저 실패한 단계 하나에 귀속**합니다(`first_failure`, `cap_lab/scorers.py:322`). 이렇게 하면 단계별 수를 더한 값이 실행 수와 같아지고, 표를 읽을 때 같은 실행을 두 번 세지 않습니다.

이 규칙에도 한계가 있습니다. 뒤 단계가 앞 단계와 무관하게 따로 실패해도 그 실행은 앞 단계로만 세어집니다. 그래도 규칙을 미리 정해 두었기 때문에, 결과를 본 뒤 유리한 세는 법을 고를 수는 없습니다.

In [ ]:
# [따라하기] 검사와 단계의 대응, 첫 실패를 찾는 순서 (cap_lab/scorers.py:26, :28)
stage_rows = []
for check, stage in STAGE_OF.items():
    stage_rows.append({"검사": check, "단계": stage, "첫 실패 순서에 들어가는가": stage in STAGE_ORDER})
stage_rows.append({"검사": "contamination_free", "단계": "전체", "첫 실패 순서에 들어가는가": False})
show_table(stage_rows)
print("첫 실패를 찾는 순서:", " -> ".join(STAGE_ORDER))

**먼저 예측하세요.** 아래 셀의 실행 하나는 S3(`vague_not_frozen`)와 S5(`frozen_ok`)가 함께 실패했고, R1(`no_blind_retry`)도 실패했습니다. `trigger_ok`와 `goal_preserved`는 None입니다. `first_failure`는 무엇을 돌려줄까요? R1은 결과에 영향을 줄까요? 모든 False를 None으로 바꾸면 무엇이 나올까요? 적은 뒤 실행하고, 주석의 바꿔 보기를 직접 해 봅니다.

In [ ]:
# [바꿔보기] 실행 하나의 점수로 첫 실패 단계를 구한다. 값을 바꿔 결과가 어떻게 바뀌는지 보세요 (호출 0회)
one_run = [
    Score("trigger_ok", None, "skill_read 기대가 없는 사례라 해당 없음"),
    Score("tool_selection_ok", True, ""),
    Score("args_valid", True, ""),
    Score("goal_preserved", None, "goal_verbatim 기대가 없는 사례라 해당 없음"),
    Score("vague_not_frozen", False, "모호한 답이 그대로 확정 명세가 됨"),
    Score("frozen_ok", False, "확정 여부가 기대와 다름"),
    Score("evidence_linked", None, "evidence_linked 기대가 없는 사례라 해당 없음"),
    Score("no_blind_retry", False, "거부된 도구를 같은 인자로 다시 부름"),
]
print("첫 실패 단계:", first_failure(one_run))
# 바꿔 보기 1: args_valid를 False로 바꾸면?
# 바꿔 보기 2: False를 모두 None으로 바꾸면? (실패가 없으면 None이다)
# 바꿔 보기 3: no_blind_retry만 False로 남기면? (R 단계는 첫 실패 순서에 없다)

### 손으로 만든 기록으로 병목 세기

진짜 측정은 7절에서 합니다. 그 전에 **손으로 만든 기록**으로 `attribute()`(`cap_lab/experiment.py:691`)가 무엇을 세는지 봅니다. 손으로 만든 기록은 실행기가 쓰는 로컬 사본 한 줄(`cap_lab/experiment.py:322`의 `_mirror_line`)과 같은 키를 갖되, 숫자는 사람이 정했습니다. 측정 결과가 아니라는 것을 잊지 않도록 사례 이름을 `손-01`처럼 붙이고, 파일은 운영체제의 임시 폴더에만 씁니다.

`attribute()`는 줄마다 세 가지 중 하나로 셉니다.

- `first_failure`가 있으면 그 단계에 1을 더합니다.
- 첫 실패가 없는데 실행 오류(`error`)가 있으면 `unchecked`(확인 불가)에 더합니다. 오류로 중간에 멈춘 실행은 검사 값이 모두 None이 되기 때문입니다(`cap_lab/scorers.py:306-311`).
- 둘 다 아니면 `no_failure`에 더합니다.

아래 도우미 셀은 이 손으로 만든 기록을 쓰는 함수를 정의합니다. 4절과 5절에서도 같은 함수를 씁니다.

In [ ]:
# [따라하기] '손으로 만든 기록'을 쓰는 도우미 (호출 0회, 운영체제 임시 폴더에만 쓴다)
HAND_DIR = Path(tempfile.mkdtemp(prefix="day04-07-hand-"))
HAND_CASES = ("손-01", "손-02", "손-03", "손-04")


def hand_line(variant: str, repeat: int, case_id: str, *, checks: dict, model_calls: int, tool_calls: int,
              latency_s: float, error: str | None = None) -> dict:
    '''손으로 만든 기록 한 줄. 실행기의 로컬 사본 한 줄(cap_lab/experiment.py:322)과 같은 키 가운데
    compare()와 attribute()가 읽는 키를 채운다.

    checks에 적지 않은 참/거짓 검사는 통과(True)로 둔다. error가 있으면 score_run(cap_lab/scorers.py:306)처럼
    참/거짓 검사를 모두 None(확인 불가)으로 바꾼다. 숫자 지표는 그대로 둔다.
    '''
    values = {name: True for name in BOOLEAN_SCORES}
    values.update(checks)
    if error:
        values = {name: None for name in values}
    scores = [Score(name, value, "손으로 만든 값") for name, value in values.items()]
    scores += [
        Score("model_calls", float(model_calls), "손으로 만든 값"),
        Score("tool_calls", float(tool_calls), "손으로 만든 값"),
        Score("latency_s", float(latency_s), "손으로 만든 값"),
    ]
    return {
        "run_name": f"{variant}-r{repeat}", "variant": variant, "repeat": repeat, "case_id": case_id,
        "split": "dev", "stage": "-", "kind": "손으로 만든 기록",
        "scores": [asdict(score) for score in scores],
        "first_failure": first_failure(scores),
        "error": error,
    }


def write_lines(path: Path, lines: list[dict]) -> Path:
    '''줄 목록을 JSON Lines 파일 하나로 쓴다. 실행기처럼 한글을 그대로 쓴다.'''
    path.write_text("".join(json.dumps(line, ensure_ascii=False) + "\n" for line in lines), encoding="utf-8")
    return path


def attribution_rows(counts: dict[str, int]) -> list[dict]:
    '''attribute() 결과를 표로 편다. 비중은 확인한 실행(확인 불가 제외) 가운데 몫이다.'''
    checked = sum(count for key, count in counts.items() if key != "unchecked")
    rows = []
    for key, count in counts.items():
        if key == "unchecked":
            share = "비중 계산에서 제외"
        elif checked == 0:
            share = "확인한 실행 없음"
        else:
            share = round(count / checked, 3)
        rows.append({"첫 실패": key, "실행 수": count, "확인한 실행 중 비중": share})
    return rows


print("손으로 만든 기록 폴더(임시):", HAND_DIR)

In [ ]:
# [따라하기] 손으로 만든 기준선 12회(4 사례 x 3 반복)의 첫 실패를 센다 (호출 0회)
# (사례, 반복, 실패시킨 검사, 실행 오류). 적지 않은 검사는 모두 통과다.
BASELINE_SPEC = [
    ("손-01", 1, {"trigger_ok": False}, None),
    ("손-01", 2, {}, None),
    ("손-01", 3, {}, None),
    ("손-02", 1, {"goal_preserved": False}, None),
    ("손-02", 2, {"goal_preserved": False, "frozen_ok": False}, None),            # S2와 S5가 함께 실패
    ("손-02", 3, {}, None),
    ("손-03", 1, {"vague_not_frozen": False, "frozen_ok": False}, None),          # S3와 S5가 함께 실패
    ("손-03", 2, {"vague_not_frozen": False}, None),
    ("손-03", 3, {"vague_not_frozen": False}, None),
    ("손-04", 1, {"vague_not_frozen": False}, None),
    ("손-04", 2, {"evidence_linked": False, "no_blind_retry": False}, None),      # S6과 R1이 함께 실패
    ("손-04", 3, {}, "RateLimitError: 429 (손으로 만든 오류)"),
]
baseline_lines = [
    hand_line("손-기준선", repeat, case_id, checks=checks, model_calls=6, tool_calls=4, latency_s=40.0, error=error)
    for case_id, repeat, checks, error in BASELINE_SPEC
]
baseline_path = write_lines(HAND_DIR / "손-기준선-r1.jsonl", baseline_lines)

hand_counts = attribute([baseline_path])
show_table(attribution_rows(hand_counts))
print("실행 수 합계:", sum(hand_counts.values()), "(손으로 만든 기록, 측정 결과가 아님)")

**관찰할 것.**

- S3가 4회로 가장 많습니다. `손-03`의 1회차는 S3와 S5가 함께 실패했지만 S3 하나로만 세었고, `손-02`의 2회차도 S2 하나로만 세었습니다.
- `손-04`의 2회차에서 실패한 R1(`no_blind_retry`)은 첫 실패 순서에 없으므로 S6으로 세었습니다. R 단계는 따로 보고합니다.
- 오류로 끝난 1회는 `unchecked`입니다. 실패로도 통과로도 세지 않고, 비중을 낼 때도 분모에서 뺍니다. 그래서 S3의 비중은 4/12가 아니라 4/11입니다.
- 모든 칸의 합은 12, 곧 실행 수입니다. 실행 하나가 정확히 한 칸에 들어갑니다.

### 판단 모델을 둘 자리의 세 조건 (P3)

가장 많이 무너지는 단계를 찾았다고 곧바로 Jev를 넣지 않습니다. 세 조건을 모두 따집니다.

| 조건 | 무엇으로 확인하는가 | 손으로 만든 기준선에서 |
|---|---|---|
| 1. 최초 실패 비중이 가장 크다 | `attribute()`의 수 | S3 (확인한 11회 중 4회) |
| 2. 판단 형태의 문제다 | 문제를 읽고 사람이 논증한다 | "이 완료 기준을 실행 결과로 참/거짓을 가를 수 있는가"에는 미리 적어 둘 정답 문자열이 없다 |
| 3. 그것을 푸는 결정적 규칙이 없다 | 규칙 후보를 세우고 반례로 깨 본다 | "숫자가 있으면 명확하다"는 "2배 더 보기 좋으면 완료"에서 깨지고, "숫자가 없으면 모호하다"는 "모든 단위 테스트가 통과하면 완료"에서 깨진다 |

코드로 계산되는 것은 1번뿐입니다. 2번과 3번은 글로 적는 논증이고, 3번은 반례를 실제로 들어야 합니다. 규칙으로 풀리는 문제에 판단 모델을 쓰면 비용과 변동만 늘어납니다. P2는 그런 예를 일부러 남겨 두었습니다. **재시도 여부는 Jev가 정하지 않습니다.** 서버가 구조화된 오류 코드와 재시도 가능 여부를 돌려주므로 코드 규칙으로 충분합니다.

### H1은 S3에, H2는 S2에 둔 이유

H1과 H2는 측정 전에 세운 **가설**입니다(P2). 기준선 측정이 그 가설을 확인하거나 반박합니다.

- **H1 (S3, 완료 기준의 테스트 가능성).** 위 표의 2번과 3번이 그대로 근거입니다. `dev-05`의 "분류가 잘 되면 완료"와 elicitation으로 받는 "샘플 문의 50건 중 담당 분류와 일치하는 문의가 45건 이상이면 완료"의 차이는 글자 규칙으로 가르기 어렵습니다.
- **H2 (S2, 목표 유지).** 사용자의 목표는 문장 속에 섞여 옵니다("목표는 「...」입니다", "「...」 이게 제 목표 문장이에요"). 검사(`goal_preserved`)는 사례에 적힌 정답 문자열과 비교하지만, 실행 중의 게이트에게는 정답 문자열이 없습니다. 모델이 목표를 '명세 만들기'처럼 다른 일로 바꿔 적었는지, 조사만 다듬었는지는 판단입니다. "goal은 사용자 메시지의 부분 문자열이어야 한다" 같은 규칙은 사용자가 따옴표를 쓰지 않거나 모델이 조사 하나를 고치면 깨집니다.

**기준선이 가설과 어긋나면 어떻게 하는가.** 병목의 근거가 되는 기준선은 저자가 dev 10 사례 x 5 반복으로 돌리는 A입니다. 그 기준선에서 첫 실패가 가장 많은 단계가 S2나 S3가 아니면, 게이트 위치를 바꾸기 전에 저자에게 묻습니다(P3). 7절의 수업 기준선은 S2와 S3를 재려고 고른 4 사례라 S1과 S6이 아예 실패할 수 없으므로, 가설을 뒤집는 근거가 되지 못합니다. 수업에서 세는 것은 읽는 법을 익히는 연습입니다.

## 3. 두 게이트: Jev에게 무엇을 묻는가

두 게이트는 같은 Decision API를 쓰지만 **사는 곳이 다릅니다.** H1은 MCP 서버(자식 프로세스) 안에 있고, H2는 Agent의 미들웨어로 이 노트북 프로세스 안에 있습니다. 이 차이는 직접 해보기에서 질문 문구를 바꿀 때 중요해집니다.

### H1: 서버의 완료 기준 게이트 (S3, `--gate h1`)

1. 모델이 `record_answer(field="acceptance_criteria", answer=...)`를 부릅니다.
2. 서버는 도구 본문을 실행하기 전에 `Resolve(assess)`로 판정합니다(`cap_lab/server/server.py:224`). `TestabilityGate.check`가 Jev에 Score 질문 `testability`를 한 번 묻습니다(`cap_lab/server/gate.py:67-83`). 같은 문장은 다시 묻지 않습니다. MRTR에서는 라운드마다 판정 함수가 다시 실행되기 때문입니다(`cap_lab/server/gate.py:63-69`).
3. 점수가 임계값 **이상**이면 모델이 준 답을 그대로 기록합니다. 근거 이벤트에 출처 `agent`가 남습니다(`cap_lab/server/server.py:283`).
4. 점수가 임계값 **미만**이면 서버가 MRTR(`Elicit`)로 **사용자에게 직접** 묻습니다(`cap_lab/server/server.py:235-246`). 이 질문과 답은 도구 인자에 나오지 않으므로 모델이 보거나 채울 수 없습니다. 사용자가 수락하면 사용자의 문장을 기록하고 출처 `elicited`와 원래 답을 남깁니다(`cap_lab/server/server.py:285`). 거절하면 `criteria_not_testable`을 돌려주고 아무것도 기록하지 않습니다(`cap_lab/server/server.py:280`).
5. Jev가 어떤 이유로든 답하지 못하면 `decision_unavailable`을 돌려주고 아무것도 기록하지 않습니다(`cap_lab/server/server.py:271`).

### H2: Agent 미들웨어 GoalGuard (S2)

1. 모델이 `start_interview(goal=...)`를 부르면, 도구가 실행되기 전에 `GoalGuard`가 가로챕니다(`cap_lab/middleware.py:61-76`). 다른 도구는 그대로 지나갑니다.
2. 첫 사용자 메시지와 `goal` 인자를 묶어 Jev에 Noul 질문 `goal_preserved`를 한 번 묻습니다(`cap_lab/middleware.py:80-84`).
3. 확률이 임계값 **이상**이면 도구를 실행합니다(`cap_lab/middleware.py:85`). **미만**이면 도구를 부르지 않고 `goal_changed`와 "사용자가 말한 목표 문장을 바꾸지 말고 그대로 goal에 넣으세요"를 돌려줍니다(`cap_lab/middleware.py:35`, `:92`). 모델은 이 결과를 읽고 다시 부를 수 있습니다.
4. Jev가 답하지 못하면 도구를 부르지 않고 `decision_unavailable`을 돌려줍니다(`cap_lab/middleware.py:88-90`).

`build_agent`는 `gate_h2=True`일 때만 GoalGuard를 미들웨어에 넣습니다(`cap_lab/agent.py:109`).

| 항목 | H1 | H2 |
|---|---|---|
| 사는 곳 | MCP 서버 자식 프로세스 | Agent 미들웨어, 노트북 프로세스 |
| 질문 | Score `testability` (단계 3개) | Noul `goal_preserved` (참/거짓 확률) |
| 묻는 때 | `record_answer`의 완료 기준마다 | `start_interview` 호출마다 |
| 통과 | 점수 >= 임계값 | 확률 >= 임계값 |
| 미달 | 사용자에게 직접 묻는다(MRTR) | 호출을 막고 `goal_changed` |
| Jev 실패 | `decision_unavailable`, 기록 없음 | `decision_unavailable`, 호출 없음 |
| 목표 지표 | `vague_criteria` 사례의 `vague_not_frozen` | `goal_verbatim` 사례의 `goal_preserved` |

In [ ]:
# [따라하기] 두 게이트가 Jev에 보내는 질문을 그대로 본다 (호출 0회)
question_rows = [
    {
        "게이트": "H1 (서버, S3)",
        "질문 종류": TESTABILITY.kind,
        "이름": TESTABILITY.name,
        "지시문": TESTABILITY.instructions,
        "기준": " / ".join(f"{index}: {level}" for index, level in enumerate(TESTABILITY.levels)),
    },
    {
        "게이트": "H2 (미들웨어, S2)",
        "질문 종류": GOAL_PRESERVED.kind,
        "이름": GOAL_PRESERVED.name,
        "지시문": GOAL_PRESERVED.instructions,
        "기준": f"참: {GOAL_PRESERVED.true} / 거짓: {GOAL_PRESERVED.false}",
    },
]
show_table(question_rows)
print("두 질문의 정의: cap_lab/server/gate.py:28 (H1), cap_lab/middleware.py:25 (H2)")

### 임계값: 0.5는 미조정 자리표시자

두 게이트의 임계값은 모두 0.5이고, **미조정**입니다. 이유는 셋입니다.

- **H2의 확률은 보정(calibration)을 가정하지 않습니다**(`cap_lab/middleware.py:5`, `cap_lab/decisions.py:6`). Jev가 0.6이라고 답해도 "60%의 경우에 맞다"는 뜻이라는 보장이 없습니다.
- **H1의 점수는 척도를 가정하지 않습니다.** 점수는 단계별 확률로 낸 기댓값이고 척도는 Decision API가 정하므로, [0, 1]이라고 가정하지 않습니다(`cap_lab/server/gate.py:3-4`). 이 노트북은 Jev Score의 실제 척도를 확인하지 않았습니다. 척도가 다르면 0.5는 아주 느슨하거나 아주 엄격한 값일 수 있습니다.
- **임계값은 라벨로만 조정합니다(P4).** 저자가 검토한 dev 라벨로 재야 하며, 측정 결과를 보고 판정이 좋아지는 값을 고르면 안 됩니다. 그렇게 고른 임계값은 그 결과에만 맞습니다.

임계값은 실행 metadata에 고정해 기록되므로(P4) 나중에 어떤 값으로 잰 결과인지 언제든 확인할 수 있습니다.

**먼저 예측하세요.** 점수가 임계값과 정확히 같으면(0.5와 0.5) 통과일까요, 미달일까요? 두 게이트가 같은 방식으로 비교할까요?

In [ ]:
# [따라하기] 임계값이 코드 어디에서 0.5로 정해지는지 확인한다 (호출 0회)
threshold_rows = [
    {"위치": "Variant.threshold_h1 기본값 (cap_lab/experiment.py:74)",
     "값": Variant.__dataclass_fields__["threshold_h1"].default},
    {"위치": "Variant.threshold_h2 기본값 (cap_lab/experiment.py:75)",
     "값": Variant.__dataclass_fields__["threshold_h2"].default},
    {"위치": "GoalGuard(threshold=...) 기본값 (cap_lab/middleware.py:55)",
     "값": inspect.signature(GoalGuard).parameters["threshold"].default},
    {"위치": "수업 B의 H1 임계값", "값": VARIANT_B.threshold_h1},
    {"위치": "수업 B의 H2 임계값", "값": VARIANT_B.threshold_h2},
]
for row in threshold_rows:
    row["상태"] = "미조정(자리표시자)"
show_table(threshold_rows)
print("서버의 --gate-threshold 기본값도 0.5이며 도움말에 '미조정 자리값'이라고 적혀 있다(cap_lab/server/server.py:394).")

In [ ]:
# [바꿔보기] 임계값과 같은 점수는 통과인가? 점수와 임계값을 바꿔 보세요 (호출 0회)
GATE_THRESHOLD = 0.5
for score in (0.49, 0.5, 0.51):
    decision = GateDecision(score=score, threshold=GATE_THRESHOLD)
    print(f"H1: testability 점수 {score} / 임계값 {GATE_THRESHOLD} -> 통과 {decision.passed}")
# H2도 같은 방식이다: probability >= threshold 이면 통과(cap_lab/middleware.py:85).
# 만약 점수가 1을 넘는 척도로 온다면? 비교 자체는 된다. 척도를 [0, 1]로 가정하지 않기 때문이다(cap_lab/server/gate.py:3).

### fail-closed: 판정이 실패하면 통과시키지 않는다

게이트가 판정기에게 답을 받지 못했을 때 "일단 통과"로 처리하면 어떻게 될까요? Jev가 느리거나 키가 틀렸거나 응답 형식이 바뀐 바로 그 순간에 게이트가 소리 없이 사라집니다. 측정에서도 문제가 됩니다. B의 실행 가운데 게이트가 실제로 동작하지 않은 실행이 '게이트를 켠 결과'로 섞여 들어갑니다.

그래서 두 게이트는 **판정 실패를 절대 통과로 바꾸지 않습니다.**

- 전송 실패, HTTP 오류, 응답 형식 오류는 `DecisionError`가 됩니다(`cap_lab/decisions.py:99-112`). 메시지에는 공급자 이름과 HTTP 상태만 담고, 키와 보낸 내용은 담지 않습니다(`cap_lab/decisions.py:81`).
- 그 밖의 예외도 종류만 남기고 실패로 다룹니다.
- Jev가 답을 거절(refused)해서 값이 비어 있어도 실패입니다.
- 실패하면 H2는 도구를 부르지 않고, H1은 아무것도 기록하지 않습니다. 둘 다 `decision_unavailable`이라는 같은 이름의 결과를 돌려줍니다.
- Claude로 대신 판정하는 길은 없습니다(`cap_lab/decisions.py:186`).

대가도 있습니다. Jev가 자주 실패하면 B의 결과가 나빠집니다. 이것은 숨길 일이 아니라 측정해야 할 일입니다. "게이트를 켠다"는 것은 판정기의 가용성까지 함께 켜는 것이기 때문입니다. 아래 셀에서 두 판정 함수의 소스를 직접 읽습니다.

In [ ]:
# [따라하기] 판정 실패를 통과로 바꾸지 않는 코드를 직접 읽는다 (호출 0회)
print("=== H2: cap_lab/middleware.py, GoalGuard._judge ===")
print(inspect.getsource(GoalGuard._judge))
print("=== H1: cap_lab/server/gate.py, TestabilityGate._judge ===")
print(inspect.getsource(TestabilityGate._judge))

**관찰할 것.**

- H2의 `_judge`는 모든 실패 갈래에서 확률 자리에 `None`을 돌려줍니다. 호출하는 쪽(`_check`, `cap_lab/middleware.py:78`)은 `None`을 보면 `decision_unavailable`로 막습니다. 어떤 갈래에도 "확률을 1로 두고 통과" 같은 줄이 없습니다.
- H1의 `_judge`는 실패하면 `GateUnavailable`을 냅니다. 서버의 `assess`가 이를 받아 실패 이유로 바꾸고, `record_answer` 본문은 그 이유를 보면 아무것도 기록하지 않습니다(`cap_lab/server/server.py:224-233`, `:269-274`).
- 둘 다 `refused`(값 없음)를 따로 검사합니다. 응답이 왔다고 판정이 있는 것은 아닙니다.
- H2의 HTTP 호출은 최대 15초가 걸릴 수 있어 스레드에서 돌립니다(`cap_lab/middleware.py:74-75`). Agent의 이벤트 루프를 막지 않기 위해서입니다.

## 4. 측정 방법: A/A 잡음 폭과 쌍체 부트스트랩

### 세 변형

| 변형 | H1 | H2 | 역할 |
|---|---|---|---|
| A | 끔 | 끔 | 기준선 |
| A2 | 끔 | 끔 | A와 **똑같은 구성을 한 번 더** 돌린다. 우연만으로 생기는 차이(잡음 폭)를 잰다 |
| B | 켬 | 켬 | 게이트를 켠 구성 |

**왜 A2가 필요한가.** 같은 Agent라도 실행마다 결과가 다릅니다. 모델의 출력이 매번 같지 않고, Jev와 네트워크의 응답 시간도 흔들립니다. B와 A의 차이가 +0.2로 나왔는데, 아무것도 바꾸지 않은 A와 A2 사이에도 ±0.2쯤의 차이가 난다면 그 +0.2는 게이트의 효과라고 말할 수 없습니다. A2는 "우연이 얼마나 큰가"를 가정하지 않고 **직접 잽니다.**

**두 게이트를 한 번에 켜는 이유와 주의점.** H1과 H2는 서로 다른 단계(S3, S2)를 겨냥하므로 목표 지표를 따로 읽습니다(P4). 다만 두 게이트가 같은 실행 안에서 서로 영향을 줄 수 있습니다. 예를 들어 GoalGuard가 `start_interview`를 한 번 막으면 그 뒤의 인터뷰 흐름과 S3 경로도 달라질 수 있습니다. 이 상호작용은 따로 분리해 재지 않으므로 결과를 보고할 때 주의사항으로 함께 적습니다.

### 사례 단위 쌍체 비교

사례마다 난이도가 크게 다릅니다. 어떤 사례는 A에서도 늘 통과하고, 어떤 사례는 B에서도 늘 실패합니다. 그래서 **사례를 자기 자신과 비교합니다.** 사례마다 A의 반복 평균과 B의 반복 평균을 내고, 그 차이(B 평균 - A 평균)를 구합니다. 통계량은 사례별 차이들의 평균입니다(`cap_lab/stats.py:30-55`). 이렇게 짝을 지으면 사례 사이의 난이도 차이가 비교에서 빠집니다.

### 부트스트랩 구간

차이의 평균 하나만으로는 그 값이 얼마나 흔들릴지 모릅니다. 그래서 **사례를 복원 추출로 다시 뽑아** 평균을 다시 내는 일을 10000번 하고(seed 0), 그 10000개 평균의 2.5% 지점과 97.5% 지점을 95% 구간 [lo, hi]로 씁니다(P5).

다시 뽑는 단위가 **실행이 아니라 사례**라는 점이 중요합니다. 같은 사례의 반복 3회는 같은 난이도를 공유하므로 서로 독립이 아닙니다. 실행 단위로 다시 뽑으면 실제보다 좁은 구간이 나와 자신 있는 척하게 됩니다. 대신 사례가 적으면 구간이 거칠어집니다. 사례가 1개면 몇 번을 다시 뽑아도 같은 사례만 나오므로 구간이 한 점으로 줄어듭니다.

### 잡음 폭

지표마다 A 대 A2 구간을 구하고, 그 구간 양 끝의 절댓값 중 큰 쪽을 **잡음 폭**으로 씁니다(`noise_band`, `cap_lab/stats.py:58-63`). 같은 구성끼리도 이만큼은 차이 날 수 있다는 폭입니다.

In [ ]:
# [따라하기] 사례 3개짜리 작은 예로 쌍체 차이를 손으로 계산하고 함수와 맞춰 본다 (호출 0회, 손으로 만든 값)
tiny_a = {"예-1": [0, 0, 1], "예-2": [1, 1, 1], "예-3": [0, 0, 0]}   # 사례별 반복 3회의 통과(1)/실패(0)
tiny_b = {"예-1": [1, 1, 1], "예-2": [1, 1, 1], "예-3": [0, 1, 0]}

# 1) 사례마다 (B 평균 - A 평균)
by_hand = {case: sum(tiny_b[case]) / len(tiny_b[case]) - sum(tiny_a[case]) / len(tiny_a[case]) for case in tiny_a}
print("사례별 (B 평균 - A 평균):", {case: round(delta, 3) for case, delta in by_hand.items()})

# 2) 통계량은 그 차이들의 평균
print("손으로 낸 통계량:", round(sum(by_hand.values()) / len(by_hand), 3))

# 3) 함수는 같은 통계량과 함께 사례를 다시 뽑아 만든 95% 구간을 준다. 10000회, seed 0이 기본값이다(cap_lab/stats.py:30).
tiny_result = paired_bootstrap_diff(tiny_a, tiny_b)
print("paired_bootstrap_diff:", {key: round(value, 3) if isinstance(value, float) else value for key, value in tiny_result.items()})

In [ ]:
# [바꿔보기] seed와 사례 수를 바꾸면 구간이 어떻게 달라지는지 본다 (호출 0회, 손으로 만든 값)
for seed in (0, 1, 2):
    interval = paired_bootstrap_diff(tiny_a, tiny_b, seed=seed)
    print(f"seed {seed}: 구간 [{interval['lo']:.3f}, {interval['hi']:.3f}]")

# 사례가 1개뿐이면 다시 뽑아도 늘 같은 사례라 lo와 hi가 같아진다.
print("사례 1개:", paired_bootstrap_diff({"예-1": tiny_a["예-1"]}, {"예-1": tiny_b["예-1"]}))

# A/A 잡음 폭: 같은 구성을 한 번 더 돌렸다고 치고 만든 값
tiny_a2 = {"예-1": [0, 1, 0], "예-2": [1, 1, 0], "예-3": [0, 0, 0]}
print("A/A 잡음 폭:", round(noise_band(tiny_a, tiny_a2), 3))
# 바꿔 보기: tiny_b의 값을 바꾸거나 사례를 하나 더 넣어 보세요. seed는 사전 등록에서 0으로 고정했습니다(P5).

**관찰할 것.**

- 사례가 3개뿐이라 구간이 거칩니다. 사례를 다시 뽑아 만들 수 있는 평균의 가짓수가 적기 때문입니다.
- seed를 바꾸면 구간 끝이 움직일 수도 있고 그대로일 수도 있습니다. 움직인다면 seed를 고정하지 않았을 때 같은 기록에서 다른 판정이 나올 수 있다는 뜻입니다. 그래서 seed 0을 사전 등록했습니다.
- 사례 1개는 lo와 hi가 같습니다. 이 구간은 불확실성에 대해 아무것도 말하지 않습니다. 수업 측정에서 목표마다 2 사례를 고른 이유입니다.

### 판정 규칙 `decide()` (P5, `cap_lab/stats.py:70`)

비교 결과가 나오면 아래 순서로 **처음 맞는 조건 하나**를 따릅니다.

| 순서 | 조건 | 판정 |
|---|---|---|
| 1 | 목표 지표의 비교 결과가 없다 | `판정 불가: 데이터 부족` |
| 2 | 목표 구간 상한 hi < 0 | `회귀` |
| 3 | 목표 구간 하한 lo <= 0 | `개선 확인 불가` |
| 4 | 가드 지표 중 비교 결과나 잡음 폭이 없는 것이 있다 | `판정 불가: 데이터 부족` |
| 5 | 가드 지표가 잡음 폭을 넘어 나빠졌다 | `교환` |
| 6 | 그 밖 | `개선` |

읽을 때 주의할 점이 셋 있습니다.

- **목표 지표는 구간으로, 가드 지표는 점 추정으로 봅니다.** 목표는 구간 전체가 0보다 위에 있어야 개선입니다. 가드는 차이(diff)가 잡음 폭을 넘는지만 봅니다.
- **나빠지는 방향이 지표마다 다릅니다.** 참/거짓 검사는 높을수록 좋으므로 diff < -잡음 폭이면 나빠진 것이고, `model_calls`, `tool_calls`, `latency_s`는 낮을수록 좋으므로 diff > 잡음 폭이면 나빠진 것입니다(`cap_lab/experiment.py:47`).
- **순서가 결과를 바꿉니다.** 3번이 4번보다 앞이므로, 목표가 0보다 확실히 오르지 않았으면 가드에 값이 없더라도 `개선 확인 불가`입니다. 목표가 올랐을 때만 가드의 값 유무를 따집니다.

In [ ]:
# [따라하기] decide()에 손으로 만든 비교 결과를 넣어 여섯 갈래를 모두 지나가 본다 (호출 0회)
target_up = {"diff": 0.42, "lo": 0.33, "hi": 0.58, "n_cases": 4}            # 목표 구간이 0보다 위에 있다
guards_fine = {
    "args_valid": {"diff": 0.0, "lo": 0.0, "hi": 0.0, "n_cases": 4},       # 높을수록 좋은 가드
    "model_calls": {"diff": 0.25, "lo": 0.0, "hi": 0.5, "n_cases": 4},     # 낮을수록 좋은 가드
}
noise_fine = {"args_valid": 0.0, "model_calls": 0.33}
lower_is_better = frozenset(LOWER_IS_BETTER)

decide_inputs = [
    ("1. 목표 비교 결과가 없다", None, guards_fine, noise_fine),
    ("2. 목표 상한 hi < 0", {"diff": -0.5, "lo": -0.67, "hi": -0.33, "n_cases": 4}, guards_fine, noise_fine),
    ("3. 목표 하한 lo <= 0", {"diff": 0.08, "lo": 0.0, "hi": 0.25, "n_cases": 4}, guards_fine, noise_fine),
    ("4. 가드의 비교 결과나 잡음 폭이 없다", target_up,
     {**guards_fine, "no_blind_retry": None}, {**noise_fine, "no_blind_retry": None}),
    ("5. 가드가 잡음 폭을 넘어 나빠졌다", target_up,
     {**guards_fine, "model_calls": {"diff": 3.25, "lo": 3.0, "hi": 3.5, "n_cases": 4}}, noise_fine),
    ("6. 그 밖", target_up, guards_fine, noise_fine),
]
decide_rows = []
for label, target, guards, noise in decide_inputs:
    verdict = decide(target, guards, noise, lower_is_better=lower_is_better)
    decide_rows.append({"상황": label, "판정": verdict["verdict"], "이유": " / ".join(verdict["reasons"])})
show_table(decide_rows)
print("다섯 판정을 모두 지났는가:", {row["판정"] for row in decide_rows} == set(ALL_VERDICTS))

### `compare()`: 로컬 사본에서 판정까지

실제 측정에서는 `decide()`를 직접 부르지 않고 `compare()`(`cap_lab/experiment.py:646`)를 부릅니다. `compare()`는 다음을 차례로 합니다.

1. A, A2, B의 로컬 사본 파일을 읽어 지표마다 {사례: 반복 값 목록}을 만듭니다. 값이 None이거나 실행 오류가 난 줄은 값에서 빼고 **확인 불가**로 셉니다. 오류가 난 줄은 숫자 지표까지 뺍니다. 중간에 멈춘 실행의 호출 수와 시간이 작게 잡혀 개선처럼 보이지 않게 하기 위해서입니다(`cap_lab/experiment.py:623-643`).
2. 지표마다 A 대 B를 쌍체 부트스트랩으로 비교합니다.
3. 지표마다 A 대 A2로 잡음 폭을 구합니다.
4. 사전 등록 규칙 `decide()`로 판정합니다. 가드 목록은 받은 그대로 넘기며, 사전 등록 목록은 `prereg_guards(목표)`가 만듭니다(`cap_lab/experiment.py:686`).
5. 판정, 이유, 지표별 비교와 잡음 폭, **값이 없는 가드 목록**(`guards_without_data`), 변형별 **확인 불가 수**(`unchecked`)를 돌려줍니다.

아래에서는 손으로 만든 A와 A2 한 쌍에 B 다섯 가지를 맞춰 봅니다. 목표 지표는 `goal_preserved`이고, A와 A2의 목표 통과 수는 사례별로 [2, 2, 1, 2]와 [2, 1, 2, 2]입니다(반복 3회 중).

| B | 목표 통과 수 (사례별, 3회 중) | 호출 수 | 그 밖 |
|---|---|---|---|
| B-개선 | [3, 3, 3, 3] | A와 비슷 | - |
| B-교환 | [3, 3, 3, 3] | 실행마다 3회 더 | - |
| B-회귀 | [0, 0, 0, 1] | A와 비슷 | - |
| B-혼합 | [3, 2, 1, 2] | A와 비슷 | - |
| B-가드없음 | [3, 3, 3, 3] | A와 비슷 | A, A2, B 모두 R1·R2 사례가 없어 `no_blind_retry`, `retry_bounded`가 None |

**먼저 예측하세요.** 다섯 B가 각각 어떤 판정을 받을지 적습니다. 특히 B-개선과 B-가드없음은 목표 지표 값이 똑같습니다. 판정도 같을까요?

In [ ]:
# [따라하기] 손으로 만든 A, A2 기록을 쓴다. 목표 지표는 goal_preserved (호출 0회)
def write_hand_variant(variant: str, *, target: str, passes: list[int], model_calls: list[list[int]],
                       tool_calls: list[list[int]], latency_s: list[list[float]],
                       extra_checks: dict | None = None, errors: dict | None = None) -> list[Path]:
    '''변형 하나의 손으로 만든 기록을 반복마다 파일 하나(<변형>-r<반복>.jsonl)로 HAND_DIR에 쓴다.

    passes[i]: 사례 i에서 목표 검사가 통과한 반복 수. 앞 반복부터 그 수만큼 True, 나머지는 False다.
    model_calls, tool_calls, latency_s: [사례][반복] 표.
    extra_checks: 모든 실행에 덮어쓸 검사 값(예: R1 사례가 없어 no_blind_retry가 None).
    errors: {(사례, 반복): 오류 문자열}. 그 실행은 참/거짓 검사가 모두 확인 불가가 된다.
    반환: 반복 순서의 파일 경로 목록.
    '''
    paths = []
    for repeat in (1, 2, 3):
        lines = []
        for index, case_id in enumerate(HAND_CASES):
            checks = {target: repeat <= passes[index], **(extra_checks or {})}
            lines.append(hand_line(
                variant, repeat, case_id, checks=checks,
                model_calls=model_calls[index][repeat - 1],
                tool_calls=tool_calls[index][repeat - 1],
                latency_s=latency_s[index][repeat - 1],
                error=(errors or {}).get((case_id, repeat)),
            ))
        paths.append(write_lines(HAND_DIR / f"{variant}-r{repeat}.jsonl", lines))
    return paths


def plus(table: list[list[float]], amount: float) -> list[list[float]]:
    '''[사례][반복] 표의 모든 칸에 같은 값을 더한 새 표.'''
    return [[value + amount for value in row] for row in table]


HAND_TARGET = "goal_preserved"
# [사례][반복] 표. A와 A2는 같은 구성이라 비슷하지만 똑같지는 않다(우연).
A_CALLS = [[6, 7, 6], [7, 7, 8], [5, 6, 6], [6, 6, 7]]
A2_CALLS = [[7, 7, 6], [7, 8, 8], [6, 5, 5], [6, 7, 7]]
B_CALLS = [[7, 6, 7], [7, 8, 7], [6, 6, 6], [7, 6, 7]]
A_TOOLS = [[4, 5, 4], [5, 5, 6], [4, 4, 4], [4, 5, 5]]
A2_TOOLS = [[5, 4, 4], [5, 6, 5], [4, 3, 4], [5, 5, 4]]
B_TOOLS = [[5, 4, 5], [5, 5, 6], [4, 4, 4], [5, 4, 5]]
A_SECONDS = [[41, 44, 39], [52, 47, 50], [35, 38, 36], [44, 40, 42]]
A2_SECONDS = [[46, 40, 43], [49, 55, 48], [38, 33, 39], [41, 47, 45]]
B_SECONDS = [[43, 45, 41], [51, 50, 52], [37, 39, 36], [45, 43, 42]]

hand_a = write_hand_variant("손-A", target=HAND_TARGET, passes=[2, 2, 1, 2],
                            model_calls=A_CALLS, tool_calls=A_TOOLS, latency_s=A_SECONDS)
hand_a2 = write_hand_variant("손-A2", target=HAND_TARGET, passes=[2, 1, 2, 2],
                             model_calls=A2_CALLS, tool_calls=A2_TOOLS, latency_s=A2_SECONDS)

summary_rows = []
for index, case_id in enumerate(HAND_CASES):
    summary_rows.append({
        "사례": case_id,
        "A 목표 통과": f"{[2, 2, 1, 2][index]}/3", "A2 목표 통과": f"{[2, 1, 2, 2][index]}/3",
        "A 호출 평균": round(sum(A_CALLS[index]) / 3, 2), "A2 호출 평균": round(sum(A2_CALLS[index]) / 3, 2),
        "A 시간 평균": round(sum(A_SECONDS[index]) / 3, 2), "A2 시간 평균": round(sum(A2_SECONDS[index]) / 3, 2),
    })
show_table(summary_rows)

In [ ]:
# [따라하기] B 다섯 가지를 쓰고 compare()로 판정한다. 가드는 사전 등록 P6 목록 그대로다 (호출 0회)
NO_R_CASES = {"no_blind_retry": None, "retry_bounded": None}   # R1·R2 사례가 없으면 두 검사는 값이 없다
hand_a_no_r = write_hand_variant("손-A-R없음", target=HAND_TARGET, passes=[2, 2, 1, 2], model_calls=A_CALLS,
                                 tool_calls=A_TOOLS, latency_s=A_SECONDS, extra_checks=NO_R_CASES)
hand_a2_no_r = write_hand_variant("손-A2-R없음", target=HAND_TARGET, passes=[2, 1, 2, 2], model_calls=A2_CALLS,
                                  tool_calls=A2_TOOLS, latency_s=A2_SECONDS, extra_checks=NO_R_CASES)
hand_b = {
    "B-개선": write_hand_variant("손-B-개선", target=HAND_TARGET, passes=[3, 3, 3, 3],
                               model_calls=B_CALLS, tool_calls=B_TOOLS, latency_s=B_SECONDS),
    "B-교환": write_hand_variant("손-B-교환", target=HAND_TARGET, passes=[3, 3, 3, 3],
                               model_calls=plus(B_CALLS, 3), tool_calls=plus(B_TOOLS, 3), latency_s=B_SECONDS),
    "B-회귀": write_hand_variant("손-B-회귀", target=HAND_TARGET, passes=[0, 0, 0, 1],
                               model_calls=B_CALLS, tool_calls=B_TOOLS, latency_s=B_SECONDS),
    "B-혼합": write_hand_variant("손-B-혼합", target=HAND_TARGET, passes=[3, 2, 1, 2],
                               model_calls=B_CALLS, tool_calls=B_TOOLS, latency_s=B_SECONDS),
    "B-가드없음": write_hand_variant("손-B-가드없음", target=HAND_TARGET, passes=[3, 3, 3, 3], model_calls=B_CALLS,
                                 tool_calls=B_TOOLS, latency_s=B_SECONDS, extra_checks=NO_R_CASES),
}
HAND_GUARDS = prereg_guards(HAND_TARGET)

hand_results = {}
scenario_rows = []
for name, b_paths in hand_b.items():
    a_paths, a2_paths = (hand_a_no_r, hand_a2_no_r) if name == "B-가드없음" else (hand_a, hand_a2)
    result = compare(a_paths, a2_paths, b_paths, target=HAND_TARGET, guards=HAND_GUARDS)
    hand_results[name] = result
    target_comparison = result["comparisons"][HAND_TARGET]
    scenario_rows.append({
        "B": name,
        "판정": result["verdict"],
        "목표 차이": round(target_comparison["diff"], 3),
        "목표 구간": f"[{target_comparison['lo']:.3f}, {target_comparison['hi']:.3f}]",
        "값 없는 가드": ", ".join(result["guards_without_data"]) or "-",
        "첫 번째 이유": result["reasons"][0],
    })
show_table(scenario_rows)
print("다섯 판정을 모두 보았는가:", {row["판정"] for row in scenario_rows} == set(ALL_VERDICTS))

In [ ]:
# [따라하기] B-개선의 지표별 비교를 한 줄씩 본다 (호출 0회)
def shown(value):
    '''값이 없으면 '없음'으로 적어 0과 헷갈리지 않게 한다. 숫자는 소수 셋째 자리까지.'''
    return "없음" if value is None else round(value, 3)


def comparison_rows(result: dict) -> list[dict]:
    '''compare() 결과를 지표마다 한 줄로 편다: 역할, 방향, B-A 차이와 구간, 사례 수, 잡음 폭, 확인 불가 수.'''
    rows = []
    for metric, comparison in result["comparisons"].items():
        rows.append({
            "지표": metric,
            "역할": "목표" if metric == result["target"] else "가드",
            "낮을수록 좋음": metric in LOWER_IS_BETTER,
            "B-A 차이": shown(None if comparison is None else comparison["diff"]),
            "구간 하한": shown(None if comparison is None else comparison["lo"]),
            "구간 상한": shown(None if comparison is None else comparison["hi"]),
            "사례 수": 0 if comparison is None else comparison["n_cases"],
            "잡음 폭(A/A)": shown(result["noise"][metric]),
            "확인 불가 A/A2/B": "/".join(str(result["unchecked"][side][metric]) for side in ("A", "A2", "B")),
        })
    return rows


show_table(comparison_rows(hand_results["B-개선"]))

**관찰할 것.**

- **B-개선.** 목표 구간 [0.333, 0.583]이 통째로 0보다 위에 있습니다. 가드는 모두 잡음 폭 안입니다. `model_calls`는 +0.25지만 A/A 잡음 폭 0.333보다 작고, `latency_s`는 +1.333초지만 잡음 폭 2.0초보다 작습니다.
- **B-교환.** 목표는 B-개선과 똑같이 올랐지만 호출 수가 실행마다 3회 늘어 잡음 폭을 크게 넘었습니다. "더 잘하지만 더 비싸다"는 결과이고, 개선이라고 부르지 않습니다.
- **B-회귀.** 구간 상한까지 0보다 아래입니다.
- **B-혼합.** 한 사례만 좋아지고 나머지는 그대로라 구간 하한이 0에 닿습니다. `개선 확인 불가`는 "나빠졌다"가 아니라 **"좋아졌다고 말할 근거가 부족하다"**입니다.
- **B-가드없음.** 목표 값은 B-개선과 같은데 판정은 `판정 불가: 데이터 부족`입니다. R1·R2 사례가 없어 두 가드에 값이 없기 때문입니다(P5 4번). 1절의 표를 떠올리면 수업 측정이 바로 이 모양입니다.
- 각 판정의 이유 문장은 `decide()`가 만든 그대로입니다. 보고할 때는 판정과 함께 이유, 값 없는 가드, 확인 불가 수를 같이 적습니다.

In [ ]:
# [바꿔보기] B-혼합의 사례별 통과 수를 바꿔 '개선 확인 불가'가 '개선'으로 넘어가는 지점을 찾아 보세요 (호출 0회)
MY_PASSES = [3, 2, 1, 2]     # 사례 4개 각각, 반복 3회 중 목표 검사가 통과한 수. 예: [3, 3, 2, 2], [3, 3, 2, 3]
my_hand_b = write_hand_variant("손-B-내것", target=HAND_TARGET, passes=MY_PASSES,
                               model_calls=B_CALLS, tool_calls=B_TOOLS, latency_s=B_SECONDS)
my_hand_result = compare(hand_a, hand_a2, my_hand_b, target=HAND_TARGET, guards=HAND_GUARDS)
my_target = my_hand_result["comparisons"][HAND_TARGET]
print("A 목표 통과 수: [2, 2, 1, 2] | 내 B:", MY_PASSES)
print(f"목표 차이 {my_target['diff']:+.3f}, 구간 [{my_target['lo']:.3f}, {my_target['hi']:.3f}] -> {my_hand_result['verdict']}")

### 실행마다 고정해 기록하는 metadata (P4)

같은 이름의 변형이라도 모델, effort, SKILL.md, 서버 코드, 임계값이 바뀌면 다른 실험입니다. 그래서 실행기는 실행마다 아래 항목을 고정해 Langfuse에 기록합니다(`cap_lab/experiment.py:410-427`).

- `model`, `effort`: 주 모델과 깊이
- `skill_sha256`: SKILL.md의 sha256. Skill 문서를 한 글자라도 고치면 바뀝니다.
- `server_sha256`: `server.py`, `workflow.py`, `evidence.py`, `gate.py`를 이어 붙인 sha256(`cap_lab/experiment.py:44`). H1 질문(`TESTABILITY`)을 고치면 바뀝니다.
- `gate_h1`, `gate_h2`, `threshold_h1`, `threshold_h2`: 게이트와 임계값
- `jev_model`: 판정기 공급자와 모델 id. 게이트를 모두 끈 변형은 `off`입니다.
- `dataset`: 데이터셋 이름. 여기에 변형의 `metadata`(수업 측정은 `measurement: class`)와 반복 번호가 더해집니다.

**빠진 것도 있습니다.** H2 질문(`GOAL_PRESERVED`)은 `cap_lab/middleware.py`에 있는데, 이 파일은 sha256에 들어가지 않습니다. 질문 문구만 바꾸면 metadata로는 구별되지 않습니다. 직접 해보기에서 질문을 바꿀 때 변형의 `metadata`에 질문의 sha256을 직접 적는 이유입니다.

아래 셀은 실행기의 공개 함수 `run_metadata`를 그대로 불러 미리 봅니다. 판정기 객체를 만들지만 HTTP 요청은 보내지 않습니다.

In [ ]:
# [따라하기] 각 변형의 실행에 고정해 기록할 metadata를 미리 본다 (호출 0회, 접속 0회)
try:
    preview_decider = make_decider(SETTINGS)   # 판정기 객체만 만든다. 요청은 보내지 않는다.
except RuntimeError as error:
    preview_decider = None
    print("Decision API 키가 없어 B의 jev_model이 'off'로 보입니다. 측정 전에 .env를 직접 채우세요:", error)

metadata_by_variant = {
    variant.name: experiment.run_metadata(variant, SETTINGS, None, preview_decider) for variant in CLASS_VARIANTS
}
metadata_keys = list(metadata_by_variant[VARIANT_A.name])
show_table([
    {"항목": key, **{name: metadata[key] for name, metadata in metadata_by_variant.items()}}
    for key in metadata_keys
])

## 5. 판정 규칙을 깨뜨려 보기 (손으로 만든 기록)

판정 규칙은 귀찮아 보이는 단계를 일부러 넣어 두었습니다. 그 단계를 빼고 싶어지는 순간은 대개 결과를 본 직후입니다. 이 절은 그 유혹 세 가지를 손으로 만든 기록으로 직접 해 보고, 각각이 무엇을 망가뜨리는지 봅니다. 세 셀 모두 호출이 없습니다.

1. A/A 잡음 폭 없이 B와 A만 비교한다.
2. 오류로 확인하지 못한 실행(None)을 실패로 센다.
3. 결과를 본 뒤 값이 있는 가드만 골라 넘긴다.

In [ ]:
# [깨뜨리기] A/A 잡음 폭 없이 B와 A만 비교하면 (호출 0회, 손으로 만든 기록)
honest = hand_results["B-개선"]

# 1) 같은 비교 결과에 잡음 폭을 모두 0으로 넣는다. "조금이라도 나빠지면 나빠진 것"이라는 읽기다.
zero_noise = {guard: 0.0 for guard in HAND_GUARDS}
naive = decide(
    honest["comparisons"][HAND_TARGET],
    {guard: honest["comparisons"][guard] for guard in HAND_GUARDS},
    zero_noise,
    lower_is_better=frozenset(LOWER_IS_BETTER),
)
print("잡음 폭 없이:", naive["verdict"])
for reason in naive["reasons"]:
    print("  -", reason)
print("사전 등록대로(A/A 잡음 폭):", honest["verdict"])

# 2) 아무것도 바꾸지 않은 A와 A2 사이에도 차이가 있다. A2를 B 자리에 넣으면 A 대 A2 비교가 된다.
a_vs_a2 = compare(hand_a, hand_a2, hand_a2, target=HAND_TARGET, guards=list(LOWER_IS_BETTER))
for metric in LOWER_IS_BETTER:
    print(f"{metric}: A2-A 차이 {a_vs_a2['comparisons'][metric]['diff']:+.3f} | "
          f"B-A 차이 {honest['comparisons'][metric]['diff']:+.3f} | 잡음 폭 {honest['noise'][metric]:.3f}")

**관찰할 것.** 잡음 폭을 0으로 두면 B-개선이 `교환`으로 바뀝니다. 그런데 아래 줄을 보면 아무것도 바꾸지 않은 A와 A2 사이에도 `latency_s`가 +1.333초, `model_calls`가 +0.167회 차이 납니다. B와 A의 차이도 그 정도입니다. 잡음 폭 없이 읽으면 우연을 게이트의 부작용으로 잘못 돌리게 됩니다. 반대 방향도 같습니다. 잡음 폭 없이 목표 지표의 작은 상승을 개선이라고 부르면 우연을 게이트의 효과로 잘못 돌리게 됩니다. A2를 돌리는 데 드는 12회는 이 착각을 막는 비용입니다.

In [ ]:
# [깨뜨리기] 확인 불가(None)를 실패(False)로 세면 (호출 0회, 손으로 만든 기록)
# 1) B-개선과 같은 기록에 실행 오류 2회를 넣는다. 오류 난 실행은 참/거짓 검사가 모두 None이다.
HAND_ERRORS = {
    ("손-02", 2): "RateLimitError: 429 (손으로 만든 오류)",
    ("손-04", 3): "RateLimitError: 429 (손으로 만든 오류)",
}
b_with_errors = write_hand_variant("손-B-오류", target=HAND_TARGET, passes=[3, 3, 3, 3], model_calls=B_CALLS,
                                   tool_calls=B_TOOLS, latency_s=B_SECONDS, errors=HAND_ERRORS)
correct = compare(hand_a, hand_a2, b_with_errors, target=HAND_TARGET, guards=HAND_GUARDS)
print("사전 등록대로:", correct["verdict"], "| B의 목표 지표 확인 불가:", correct["unchecked"]["B"][HAND_TARGET], "회")


# 2) 잘못된 읽기: 오류 표시를 지우고 None을 False로 바꾼 사본을 만들어 같은 규칙으로 비교한다.
def none_as_failure(paths: list[Path], label: str) -> list[Path]:
    '''잘못된 변환을 보여 주기 위한 함수. 오류 표시를 지우고 참/거짓 검사의 None을 모두 False로 바꾼 사본을 쓴다.'''
    copies = []
    for path in paths:
        lines = read_lines([path])
        for line in lines:
            if line["error"]:
                line["error"] = None
                for score in line["scores"]:
                    if score["name"] in BOOLEAN_SCORES and score["value"] is None:
                        score["value"] = False
        copies.append(write_lines(HAND_DIR / f"{label}-{path.name}", lines))
    return copies


wrong = compare(hand_a, hand_a2, none_as_failure(b_with_errors, "None을실패로"), target=HAND_TARGET, guards=HAND_GUARDS)
print("None을 실패로 센 뒤:", wrong["verdict"])
for reason in wrong["reasons"]:
    print("  -", reason)

# 3) 판정은 P5 3번에서 먼저 멈췄다. 가드도 보면, 높을수록 좋은 가드가 diff < -잡음 폭이면 나빠진 것이다(cap_lab/stats.py:66-67).
dragged = [
    guard for guard in HAND_GUARDS
    if guard not in LOWER_IS_BETTER and wrong["comparisons"][guard] is not None
    and wrong["comparisons"][guard]["diff"] < -wrong["noise"][guard]
]
print(f"잡음 폭을 넘어 나빠진 것처럼 보이게 된 참/거짓 가드 {len(dragged)}개:", dragged)

**관찰할 것.**

- 사전 등록대로 읽으면 오류 2회는 확인 불가로 빠지고 판정은 `개선`입니다. 확인 불가 수(2회)는 판정과 함께 보고합니다.
- None을 False로 바꾸면 판정이 `개선 확인 불가`로 떨어집니다. 오류가 난 두 실행이 목표 검사의 실패로 바뀌어 그 두 사례의 차이가 0이 되었고, 구간 하한이 0에 닿았기 때문입니다. 판정은 P5 3번에서 먼저 멈추지만, 마지막 줄을 보면 참/거짓 가드 10개도 모두 잡음 폭을 넘어 나빠진 것처럼 보입니다. 429는 API 사용량 제한에 걸린 오류일 뿐 게이트가 한 일이 아닌데, 그 오류가 게이트의 실패와 부작용으로 둔갑했습니다.
- 반대 방향의 함정도 있습니다. 오류로 일찍 멈춘 실행은 호출 수와 시간이 작게 잡힙니다. 그 숫자를 그대로 세면 B가 더 싸고 빨라진 것처럼 보입니다. 그래서 `compare()`는 오류가 난 줄을 숫자 지표까지 통째로 뺍니다(`cap_lab/experiment.py:626-629`).
- 오류 줄을 지우는 것도 답이 아닙니다. 지우면 확인 불가 수가 사라져 "몇 번을 확인하지 못했는가"를 보고할 수 없습니다. 로컬 사본은 오류 줄을 남기고(P4), 비교만 그 줄을 뺍니다.

In [ ]:
# [깨뜨리기] 결과를 본 뒤 값이 있는 가드만 골라 넘기면 (호출 0회, 손으로 만든 기록)
registered = hand_results["B-가드없음"]
print("사전 등록 P6 가드 그대로:", registered["verdict"])
print("  값 없는 가드:", registered["guards_without_data"])

# 값이 없는 가드를 빼고 같은 기록을 다시 판정한다.
picked_guards = [guard for guard in HAND_GUARDS if guard not in registered["guards_without_data"]]
picked = compare(hand_a_no_r, hand_a2_no_r, hand_b["B-가드없음"], target=HAND_TARGET, guards=picked_guards)
print("값 있는 가드만 골라서:", picked["verdict"])
print("  빠진 가드:", sorted(set(HAND_GUARDS) - set(picked_guards)))

**관찰할 것.** 같은 기록인데 가드 목록만 바꾸니 `판정 불가: 데이터 부족`이 `개선`으로 바뀌었습니다. 빠진 두 가드는 "권한 거부 뒤에 같은 호출을 되풀이하지 않는가", "일시 오류 뒤 재시도가 상한 안인가"입니다. 이 기록은 그 둘에 대해 아무것도 모릅니다. 모르는 것을 문제없다고 읽은 셈입니다. P6이 가드 목록을 "목표가 아닌 모든 참/거짓 검사와 숫자 셋"으로 미리 못 박은 이유가 이것이고, `prereg_guards()`는 그 목록을 코드로 만듭니다.

**먼저 예측하세요.** 1절의 표에서 수업 측정의 4 사례는 `trigger_ok`, `evidence_linked`, `no_blind_retry`, `retry_bounded`, `side_effect_stable` 다섯 가드에 값이 없었습니다. 그렇다면 8절에서 `compare(..., guards=prereg_guards(목표))`가 낼 수 있는 판정은 다섯 가운데 어느 것입니까? 목표 구간 하한이 0보다 크게 나오면 무엇이 나옵니까? 목표 구간이 0을 걸치면 무엇이 나옵니까? 답을 적어 두고 8절에서 맞춰 봅니다.

## 6. Langfuse 준비: 규모 고지, 인증, 데이터셋

여기서부터 외부 서비스에 접속합니다. 측정 기록이 어디에 어떻게 남는지 먼저 정리합니다.

| 무엇 | 어디에, 어떤 모양으로 | 코드 |
|---|---|---|
| 데이터셋 항목 | `day04_new-capability-v1`. 항목 ID는 사례 ID. input에는 id와 turns만, expected_output에는 expect, metadata에는 split, stage, kind | `cap_lab/experiment.py:89-122` |
| 데이터셋 실행 | 실험 이름 `day04_new-capability-v1:<변형>`, 실행 이름 `<변형>-r<반복>`. 반복마다 실행 하나 | `cap_lab/experiment.py:542-554` |
| 항목 점수 | 검사 값. 참/거짓은 BOOLEAN 1/0, 숫자는 NUMERIC. **None은 보내지 않는다** | `cap_lab/experiment.py:358-369` |
| 실행 점수 | 검사마다 통과율, 95% Wilson 구간, 확인 불가 수 | `cap_lab/experiment.py:372-390` |
| 실행 metadata | 4절의 표에 반복 번호를 더한 것 | `cap_lab/experiment.py:410-427`, `:553` |
| 추적(trace) | Claude 호출과 도구 호출. 실행기가 Langfuse 클라이언트를 **직접 만들 때만** LangChain 콜백이 붙는다 | `cap_lab/experiment.py:519`, `:532` |
| 로컬 사본 | `outputs/class/runs/<실행 이름>.jsonl`. 오류 항목을 포함한 모든 항목 | `cap_lab/experiment.py:537-538` |
| 작업 폴더 | `outputs/class/work/<실행 이름>/<사례>/`. 세션 DB와 근거 파일 | `cap_lab/experiment.py:506` |

두 가지를 기억합니다.

- **Langfuse의 항목 점수에는 None이 없으므로 빈칸이 생깁니다.** 확인 불가 수는 실행 점수의 comment와 metadata에 적힙니다. 판정은 오류 항목까지 모두 담은 **로컬 사본**으로 냅니다.
- **실행기에 Langfuse 클라이언트를 넘기지 않습니다.** 넘기면 실행기가 "클라이언트를 직접 만들지 않았다"고 보고 LangChain 추적 콜백을 붙이지 않습니다(`cap_lab/experiment.py:519`). 이 노트북의 클라이언트는 인증 확인과 데이터셋 동기화에만 씁니다.

**외부로 나가는 것.** Langfuse는 프롬프트와 도구 입출력을 받고, Decision API는 사용자 답변과 goal 문장을 받습니다(`docs/DESIGN.md` 9절). 사례의 문장은 모두 지어낸 업무이고 실제 고객 정보는 없습니다.

**인증 확인이 먼저입니다(P4).** 인증이 통과하지 않으면 측정 셀은 모두 멈춥니다. 기록 없이 측정만 하는 대체 경로는 없습니다. Langfuse 기록이 없는 측정은 이 사전 등록의 측정이 아니기 때문입니다. 혹시 이 셀을 건너뛰어도 `run_experiment`가 실행 전에 인증을 다시 확인하므로(`cap_lab/experiment.py:521`) 인증 없이 측정이 시작되지는 않습니다. 참고로 설치된 Langfuse SDK의 `auth_check()`는 401일 때 False를 돌려주지 않고 예외를 냅니다(`.venv/lib/python3.13/site-packages/langfuse/_client/client.py:3491`). 그래서 아래 셀은 두 경우를 모두 실패로 다룹니다.

다음 셀은 이 절과 7절 전체의 호출 규모를 먼저 알립니다. 그 셀 자체는 아무것도 호출하지 않습니다.

In [ ]:
# [따라하기] 여기부터 Langfuse에 접속하고 7절에서 유료 측정을 한다. 전체 규모를 먼저 알린다
announce("수업 측정 전체: A, A2, B 각각 사례 4 x 반복 3", model_calls=324, jev_calls=72, runs=36)
print("이 셀은 알리기만 하고 아무것도 호출하지 않습니다. 실제 호출은 7절의 A, A2, B 셀에서 일어납니다.")
print("Claude 호출은 실행당 최대", CLAUDE_CALLS_PER_RUN[1], "회, Jev 호출은 게이트 하나가 실행당 최대",
      JEV_CALLS_PER_GATE_RUN, "회로 잡은 추정 상한입니다.")
print("실행은 한 번에", MAX_CONCURRENCY, "개씩 순서대로 돌기 때문에 오래 걸릴 수 있습니다. 실행 시간은 미리 재지 않았습니다.")

In [ ]:
# [따라하기] Langfuse 인증 확인: 실패하면 여기서 멈춘다 (모델 호출 0회, 대체 경로 없음)
from langfuse import Langfuse

from cap_lab.settings import require

# 1) 키 두 개가 비어 있으면 이름만 대고 멈춘다. 값은 출력하지 않는다(cap_lab/settings.py:81).
require(SETTINGS, "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY")

# 2) 실행기와 같은 인자로 클라이언트를 만든다(cap_lab/experiment.py:442). 키는 Day04_new/.env에서만 읽는다.
langfuse_client = Langfuse(
    public_key=SETTINGS.langfuse_public_key,
    secret_key=SETTINGS.langfuse_secret_key,
    base_url=SETTINGS.langfuse_host or None,
    environment=SETTINGS.langfuse_environment,
)

# 3) 인증 확인. False가 오든 예외가 나든 실패다. 예외 메시지는 출력하지 않고 종류만 남긴다.
try:
    LANGFUSE_READY = bool(langfuse_client.auth_check())
    auth_failure = "" if LANGFUSE_READY else "auth_check가 False를 돌려주었습니다"
except Exception as error:
    LANGFUSE_READY = False
    auth_failure = type(error).__name__

# 4) 실패하면 멈춘다. 이후 측정 셀도 stop_unless_langfuse_ready()로 다시 막힌다.
if not LANGFUSE_READY:
    raise RuntimeError(
        f"Langfuse 인증 확인에 실패했습니다({auth_failure}). 6~8절의 측정 셀을 실행하지 않습니다. "
        "Day04_new/.env의 LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_HOST를 직접 확인하고 "
        "커널을 재시작한 뒤 1절부터 다시 실행하세요. 기록 없이 측정만 하는 대체 경로는 없습니다."
    )
print("Langfuse 인증: 통과 | 환경(environment):", SETTINGS.langfuse_environment)

### 데이터셋은 dev 사례로만 맞춘다

`sync_dataset`(`cap_lab/experiment.py:98`)은 사례 목록과 Langfuse 데이터셋 `day04_new-capability-v1`을 맞춥니다. 항목 ID가 사례 ID이고, 이미 같은 내용인 항목은 다시 쓰지 않습니다(`cap_lab/experiment.py:113-119`). 그래서 여러 번 불러도 데이터셋이 불어나지 않습니다.

이 노트북은 **dev 10 사례만** 맞춥니다. holdout 항목은 수업에서 Langfuse에 올리지 않습니다. 실행기는 실행하는 사례를 실행 직전에 다시 맞추므로(`cap_lab/experiment.py:522`), holdout 항목은 저자가 holdout을 여는 실행이 올립니다. 데이터셋 항목의 input에는 사용자 발화만 있고 기대값은 expected_output에 따로 있습니다. 에이전트는 기대값을 받지 않습니다.

In [ ]:
# [따라하기] Langfuse 데이터셋을 dev 사례 10개와 맞춘다 (모델 호출 0회, holdout은 올리지 않는다)
stop_unless_langfuse_ready()
assert {case["split"] for case in DEV_CASES} == {"dev"}, "dev가 아닌 사례가 섞였습니다. 동기화하지 않습니다."

written_ids = sync_dataset(langfuse_client, DEV_CASES)
print("데이터셋:", DATASET_NAME)
print("이번에 맞춘 사례 수:", len(DEV_CASES))
print("새로 쓰거나 고친 항목:", written_ids if written_ids else "없음(이미 같은 내용)")

## 7. 수업 측정: A, A2, B

순서는 **A → A2 → B**입니다.

- **A를 먼저 돌려** 기준선의 첫 실패를 셉니다. 게이트를 켜기 전에 병목을 읽는 것이 P3의 순서입니다.
- **A2를 B보다 먼저 돌립니다.** 잡음 폭이 B의 결과를 보기 전에 정해지므로, B를 본 뒤 "이 정도는 우연"이라고 기준을 늦출 수 없습니다.
- **B를 마지막에 돌립니다.**

각 셀은 실행 직전에 규모를 알리고, 실행기도 자기 추정을 한 줄 출력합니다. 셀이 끝날 때까지 기다립니다. 실행 하나가 오류로 끝나도 실행기는 멈추지 않고 그 항목을 오류로 남긴 뒤 다음으로 갑니다(`cap_lab/experiment.py:143`, `:168-169`). 만약 셀 자체가 중간에 끊기면(커널 중단 등) 같은 이름으로는 다시 시작할 수 없습니다. 그때는 끊긴 기록을 지우지 말고 새 이름으로 다시 돌린 뒤 두 기록을 함께 보고합니다.

**먼저 예측하세요.** 실행하기 전에 세 가지를 적습니다.

1. A의 12회 가운데 첫 실패가 S2인 실행과 S3인 실행은 각각 몇 회쯤일까요?
2. A와 A2의 `goal_preserved` 잡음 폭은 0에 가까울까요, 0.3을 넘을까요?
3. B에서 GoalGuard가 `start_interview`를 막는 일(`goal_changed`)이 있을까요? 있다면 어느 사례에서일까요?

In [ ]:
# [따라하기] A 실행: 게이트를 모두 끈 기준선 (사례 4 x 반복 3 = 12회)
stop_unless_langfuse_ready()
announce("A 실행: 게이트 끔, 사례 4 x 반복 3", model_calls=108, runs=12)
a_started = time.perf_counter()
run_names_a = await run_experiment(VARIANT_A, CLASS_CASES, CLASS_REPEATS, settings=SETTINGS, mirror_dir=CLASS_MIRROR_DIR)
A_WALL_SECONDS = time.perf_counter() - a_started

a_lines = read_lines(mirror_files(VARIANT_A.name))
print("A 실행 이름:", run_names_a)
print(f"A 셀 전체 시간: {A_WALL_SECONDS / 60:.1f}분")
print("A 로컬 사본 줄 수:", len(a_lines), "| 오류로 끝난 실행:", sum(1 for line in a_lines if line["error"]))

**관찰할 것.**

- 출력 맨 위에 `announce`의 한 줄과 실행기의 한 줄이 함께 있습니다. 실행기의 줄은 4절의 비용 표와 같은 계산입니다.
- 로컬 사본은 12줄이어야 합니다. 오류로 끝난 실행이 있어도 줄은 남습니다. 오류가 있다면 그 줄의 `error`를 읽어 원인(429, 시간 초과, 인프라 오류 등)을 적어 둡니다. 다시 돌려 덮지 않습니다.
- Langfuse에는 `class-A-r1`, `class-A-r2`, `class-A-r3` 세 데이터셋 실행이 생깁니다. `RUN_PREFIX`를 바꿨다면 그 이름으로 생깁니다.

In [ ]:
# [따라하기] A가 실제로 걸린 시간으로 A2와 B의 시간을 어림한다 (호출 0회)
a_seconds = [score_of(line, "latency_s") for line in a_lines]
a_seconds = [value for value in a_seconds if value is not None]
if a_seconds:
    print(f"A 실행 한 번의 latency_s 평균: {sum(a_seconds) / len(a_seconds):.1f}초 "
          "(에이전트를 만든 뒤부터 잰 시간, cap_lab/experiment.py:213)")
else:
    print("A의 latency_s 값이 없습니다. 로컬 사본의 error 칸을 확인하세요.")
print(f"A 셀 전체 시간: {A_WALL_SECONDS / 60:.1f}분 (MCP 서버를 띄우는 시간과 Langfuse 전송까지 포함)")
print(f"A2 어림: 약 {A_WALL_SECONDS / 60:.0f}분 (A와 같은 구성)")
print(f"B 어림: {A_WALL_SECONDS / 60:.0f}분 이상 (Jev 호출이 더해지고, 게이트가 막으면 모델이 다시 부를 수 있다)")

In [ ]:
# [따라하기] 기준선 A의 첫 실패를 센다: 수업 사례에서 본 병목 후보 (호출 0회)
a_counts = attribute(mirror_files(VARIANT_A.name))
show_table(attribution_rows(a_counts))

stage_counts = {stage: a_counts[stage] for stage in STAGE_ORDER}
top_count = max(stage_counts.values())
leaders = [stage for stage, count in stage_counts.items() if count == top_count and count > 0]
if not leaders:
    print("A에서 단계 실패가 없었습니다. 이 4 사례로는 병목 후보를 말할 수 없습니다.")
elif set(leaders) <= {"S2", "S3"}:
    print("첫 실패가 가장 많은 단계:", leaders, "- 가설 H2(S2), H1(S3)의 자리와 겹칩니다.")
else:
    print("첫 실패가 가장 많은 단계:", leaders, "- 가설 자리(S2, S3)와 다릅니다. P3에 따라 게이트를 옮기기 전에 저자에게 묻습니다.")
print("이 4 사례에서는 S1(trigger_ok)과 S6(evidence_linked)에 값이 없어 첫 실패가 될 수 없습니다(1절 표).")

**관찰할 것.**

- 표의 합은 12입니다. `unchecked`가 있으면 그만큼은 병목을 말할 근거에서 빠집니다.
- 이 표로 할 수 있는 말과 없는 말을 나눕니다. **할 수 있는 말:** "S2와 S3를 재려고 고른 4 사례에서, 게이트가 꺼져 있을 때 첫 실패가 이렇게 나뉘었다." **할 수 없는 말:** "이 Agent의 병목은 S3다." 사례를 S2와 S3에 맞춰 골랐으므로 다른 단계는 처음부터 실패할 기회가 적었습니다.
- 첫 실패가 하나도 없거나 S5에 몰렸다면, 그 사실을 그대로 적습니다. 가설과 맞지 않는 결과를 버리지 않습니다. 병목의 공식 근거는 저자의 dev 10 사례 기준선이고, 그 기준선이 가설과 어긋나면 게이트를 옮기기 전에 저자에게 묻습니다(P3).
- P3의 2번(판단 형태)과 3번(결정적 규칙 없음) 조건은 이 표가 답하지 않습니다. 2절의 논증으로 답합니다.

In [ ]:
# [깨뜨리기] 같은 변형 이름으로 다시 실행해 본다: 실행기가 시작 전에 거부해야 한다
# A 기록이 없을 때 이 셀을 실행하면 실제 측정이 시작되므로, 기록이 있는지 먼저 확인한다.
if not mirror_files(VARIANT_A.name):
    raise RuntimeError("A 기록이 아직 없습니다. 이 셀은 A를 실행한 뒤에만 의미가 있습니다. 실제 측정이 시작되지 않도록 멈춥니다.")

announce("A 이름 재사용 시도: 기록이 이미 있어 거부되어야 하므로 실제 호출 0회", model_calls=0)
try:
    await run_experiment(VARIANT_A, CLASS_CASES, CLASS_REPEATS, settings=SETTINGS, mirror_dir=CLASS_MIRROR_DIR)
except RuntimeError as error:
    print("거부됨:", error)
else:
    print("거부되지 않았습니다. 기록이 섞였을 수 있으니 여기서 멈추고 로컬 사본을 확인하세요.")

**관찰할 것.**

- 실행기는 먼저 비용 한 줄을 출력하고(`cap_lab/experiment.py:501`) 키를 확인한 뒤, 같은 이름의 기록이 있는지 보고 거부합니다(`cap_lab/experiment.py:508-513`). 이 거부는 Langfuse 접속, 판정기 생성, 모델 호출보다 앞에 있으므로 호출은 0회입니다. `announce`가 출력한 줄의 규모 칸이 비어 있는 것도 그래서입니다.
- 이어 쓰기를 허락하면 서로 다른 실행이 한 파일, 한 Langfuse 실행에 섞입니다. 덮어쓰기를 허락하면 실패한 기록이 사라집니다. 둘 다 사전 등록이 금지한 일(P4: 오류를 포함한 모든 항목을 남긴다)입니다.
- 구성을 바꿨거나 다시 재야 하면 **새 이름**을 씁니다. 직접 해보기가 그렇게 합니다.

In [ ]:
# [따라하기] A2 실행: A와 똑같은 구성을 한 번 더 (사례 4 x 반복 3 = 12회)
stop_unless_langfuse_ready()
announce("A2 실행: 게이트 끔, A와 같은 구성, 사례 4 x 반복 3", model_calls=108, runs=12)
run_names_a2 = await run_experiment(VARIANT_A2, CLASS_CASES, CLASS_REPEATS, settings=SETTINGS, mirror_dir=CLASS_MIRROR_DIR)

a2_lines = read_lines(mirror_files(VARIANT_A2.name))
print("A2 실행 이름:", run_names_a2)
print("A2 로컬 사본 줄 수:", len(a2_lines), "| 오류로 끝난 실행:", sum(1 for line in a2_lines if line["error"]))

**관찰할 것.** A2는 A와 이름만 다르고 구성이 같습니다. metadata도 변형 이름을 빼면 같아야 합니다. 그런데도 결과는 A와 조금 다를 것입니다. 그 차이가 이 측정에서 "우연"이 얼마나 큰지에 대한 유일한 근거입니다. 아래 셀에서 B를 보기 전에 잡음 폭을 먼저 확인합니다.

In [ ]:
# [따라하기] A와 A2의 차이로 지표별 잡음 폭을 미리 본다. B를 돌리기 전에 정해지는 값이다 (호출 0회)
preview_metrics = ["goal_preserved", "vague_not_frozen", "model_calls", "tool_calls", "latency_s"]
# compare()가 안에서 쓰는 함수(cap_lab/experiment.py:623)를 그대로 불러 같은 규칙(None·오류 줄 제외)으로 값을 모은다.
values_a, unchecked_a = experiment.case_values(mirror_files(VARIANT_A.name), preview_metrics, "dev")
values_a2, unchecked_a2 = experiment.case_values(mirror_files(VARIANT_A2.name), preview_metrics, "dev")

noise_rows = []
for metric in preview_metrics:
    a_vs_a2_real = paired_bootstrap_diff(values_a[metric], values_a2[metric])
    band = noise_band(values_a[metric], values_a2[metric])
    noise_rows.append({
        "지표": metric,
        "A2-A 차이": shown(None if a_vs_a2_real is None else a_vs_a2_real["diff"]),
        "사례 수": 0 if a_vs_a2_real is None else a_vs_a2_real["n_cases"],
        "잡음 폭": shown(band),
        "확인 불가 A/A2": f"{unchecked_a[metric]}/{unchecked_a2[metric]}",
    })
show_table(noise_rows)

In [ ]:
# [따라하기] B 실행: H1과 H2를 모두 켠다 (사례 4 x 반복 3 = 12회)
stop_unless_langfuse_ready()
announce("B 실행: H1과 H2 켬(임계값 0.5 미조정), 사례 4 x 반복 3", model_calls=108, jev_calls=72, runs=12)
run_names_b = await run_experiment(VARIANT_B, CLASS_CASES, CLASS_REPEATS, settings=SETTINGS, mirror_dir=CLASS_MIRROR_DIR)

b_lines = read_lines(mirror_files(VARIANT_B.name))
print("B 실행 이름:", run_names_b)
print("B 로컬 사본 줄 수:", len(b_lines), "| 오류로 끝난 실행:", sum(1 for line in b_lines if line["error"]))

**관찰할 것.** B의 실행기 한 줄에는 Jev 추정치가 들어 있습니다(A와 A2에는 0회). 판정기는 실행마다가 아니라 변형마다 한 번 만들어집니다(`cap_lab/experiment.py:514-515`). H1은 서버 자식 프로세스가 같은 `.env`로 자기 판정기를 따로 만듭니다(`cap_lab/server/server.py:212-222`). 다음 셀에서 두 게이트가 실제로 무엇을 했는지 셉니다.

In [ ]:
# [따라하기] B에서 두 게이트가 실제로 한 일을 로컬 사본에서 센다 (호출 0회)
gate_rows = []
for line in b_lines:
    for decision in line["gate_decisions"]:
        if decision["gate"] == "h2":
            jev_value = decision["probability"]
            outcome = "통과" if decision["allowed"] else decision["error"]
        else:
            jev_value = decision["score"]
            outcome = f"기록됨(출처 {decision['source']})"
        gate_rows.append({
            "사례": line["case_id"], "반복": line["repeat"], "게이트": decision["gate"],
            "Jev 값": jev_value, "임계값": decision["threshold"], "결과": outcome,
        })
if gate_rows:
    show_table(gate_rows)
else:
    print("기록된 게이트 판단이 없습니다.")

# 게이트가 막았거나 사용자가 거절해 업무 오류로 끝난 호출
blocked = Counter()
for line in b_lines:
    for call in line["calls"]:
        result = call.get("result")
        if isinstance(result, dict) and result.get("error") in ("goal_changed", "decision_unavailable", "criteria_not_testable"):
            blocked[f"{call['name']}: {result['error']}"] += 1
print("게이트가 막거나 거절된 호출:", dict(blocked) if blocked else "없음")
print("서버가 사용자에게 직접 물은 횟수(elicitation):", sum(len(line["interrupts"]) for line in b_lines))

**관찰할 것.**

- `h2` 줄의 Jev 값은 확률이고, `h1` 줄의 Jev 값은 점수입니다. 둘은 척도가 다르므로 나란히 비교하지 않습니다.
- `h1`의 출처가 `elicited`이면, 점수가 임계값 미만이라 서버가 사용자에게 직접 물었고 사용자가 새 기준을 준 경우입니다. `dev-05`에서 기대하는 경로입니다. `dev-06`은 응답이 없어 거절되므로 `record_answer: criteria_not_testable`로 세어지고, `h1` 기록은 남지 않습니다(답이 기록되지 않았으므로).
- `decision_unavailable`이 보이면 Jev가 답하지 못한 것입니다. fail-closed로 막혔고, 이것도 B의 결과에 포함됩니다. 횟수를 적어 둡니다.
- 위 내용은 기대하는 경로이지 확인된 결과가 아닙니다. 실제로 어떤 경로가 몇 번 나왔는지는 표가 말해 줍니다. 실제 모델과 실제 Jev로 이 경로들이 어떻게 나뉘는지는 이 노트북을 쓸 때 미리 확인하지 않았습니다.

## 8. 판정 읽기

이제 `compare()`로 두 목표 지표를 따로 판정합니다(P4: 두 게이트는 다른 단계를 겨냥하므로 목표 지표를 각각 따로 읽는다). 가드는 `prereg_guards(목표)`로 만든 사전 등록 목록을 그대로 씁니다. 결과는 아래 순서로 읽고, 그 순서대로 보고합니다.

1. **판정**: 다섯 가운데 무엇인가.
2. **이유**: `decide()`가 적은 문장. 어느 조건에서 멈췄는지 알려 준다.
3. **값 없는 가드**(`guards_without_data`): 무엇을 모르는가.
4. **확인 불가 수**(`unchecked`): 몇 번의 실행을 확인하지 못했는가.
5. **지표별 표**: 목표 구간, 가드 차이와 잡음 폭.

어떤 판정이 나와도 그대로 적습니다. 반복을 늘리지 않고, 가드를 고르지 않고, 게이트를 빼지 않습니다.

In [ ]:
# [따라하기] H2의 목표 지표 goal_preserved로 A, A2, B를 비교한다. 가드는 사전 등록 P6 목록 그대로다 (호출 0회)
def report(result: dict) -> None:
    '''compare() 결과를 판정, 이유, 값 없는 가드, 확인 불가 수, 지표별 표 순서로 보여 준다.'''
    target = result["target"]
    print("목표 지표:", target, "| 판정:", result["verdict"])
    for reason in result["reasons"]:
        print("  이유:", reason)
    print("  값 없는 가드:", result["guards_without_data"] if result["guards_without_data"] else "없음")
    print("  목표 지표 확인 불가 수 A/A2/B:",
          "/".join(str(result["unchecked"][side][target]) for side in ("A", "A2", "B")))
    show_table(comparison_rows(result))


class_files = {variant.name: mirror_files(variant.name) for variant in CLASS_VARIANTS}
incomplete = [name for name, files in class_files.items() if len(files) != CLASS_REPEATS]
if incomplete:
    raise RuntimeError(f"반복 {CLASS_REPEATS}회의 기록이 다 있지 않은 변형이 있습니다: {incomplete}. 7절을 먼저 끝내세요.")

h2_result = compare(
    class_files[VARIANT_A.name], class_files[VARIANT_A2.name], class_files[VARIANT_B.name],
    target="goal_preserved", guards=prereg_guards("goal_preserved"), split="dev",
)
report(h2_result)

In [ ]:
# [따라하기] H1의 목표 지표 vague_not_frozen으로 A, A2, B를 비교한다 (호출 0회)
h1_result = compare(
    class_files[VARIANT_A.name], class_files[VARIANT_A2.name], class_files[VARIANT_B.name],
    target="vague_not_frozen", guards=prereg_guards("vague_not_frozen"), split="dev",
)
report(h1_result)
print("5절의 예측과 맞춰 보세요. 두 판정:", {"goal_preserved": h2_result["verdict"], "vague_not_frozen": h1_result["verdict"]})

**관찰할 것.**

- **값 없는 가드에는 1절의 다섯 검사**(`trigger_ok`, `evidence_linked`, `no_blind_retry`, `retry_bounded`, `side_effect_stable`)가 들어 있을 것입니다. 이 사례들에서 값이 나올 수 없기 때문입니다. `args_valid`도 MCP 호출이 없는 실행만 있었다면 여기에 들어갈 수 있습니다.
- 그래서 수업 측정이 낼 수 있는 판정은 셋뿐입니다. 목표 구간 상한이 0보다 아래면 `회귀`, 구간이 0을 걸치거나 0에 닿으면 `개선 확인 불가`, 구간 전체가 0보다 위면 P5 4번에서 `판정 불가: 데이터 부족`입니다. `개선`이나 `교환`은 구조적으로 나올 수 없습니다. 이것은 실패가 아니라 4 사례 설계의 한계이며, 개선 주장은 모든 가드에 값이 있는 저자의 dev 10 사례 측정과 holdout 측정에서만 합니다.
- **`개선 확인 불가`는 정당한 결과입니다.** 목표마다 2 사례 x 3 반복이면 구간이 넓기 쉽습니다. "게이트가 효과가 없다"는 뜻이 아니라 "이 측정으로는 효과를 확인할 수 없다"는 뜻입니다.
- 판정이 `판정 불가`일 때 표에서 목표 구간이 0보다 위에 있는 것을 보고 "사실상 개선"이라고 쓰고 싶어질 수 있습니다. 쓰지 않습니다. 그것이 5절의 세 번째 깨뜨리기와 같은 읽기입니다.
- 두 게이트를 함께 켰으므로 한쪽 목표 지표의 변화에 다른 게이트가 영향을 주었을 수 있습니다(4절의 주의사항). 보고할 때 함께 적습니다.

In [ ]:
# [따라하기] 참고: 변형별 목표 검사 통과율과 Wilson 95% 구간. 판정이 아니다 (호출 0회)
def flat_rows(variant: Variant) -> list[dict]:
    '''로컬 사본 줄을 pass_rate_table(cap_lab/notebook.py:182)이 읽는 모양({variant, 검사 이름: 값})으로 편다.

    오류로 끝난 실행은 검사 값이 이미 None이므로 통과율에서 확인 불가로 세어진다.
    '''
    rows = []
    for line in read_lines(mirror_files(variant.name)):
        row = {"variant": variant.name, "case_id": line["case_id"]}
        row.update({score["name"]: score["value"] for score in line["scores"]})
        rows.append(row)
    return rows


all_rows = [row for variant in CLASS_VARIANTS for row in flat_rows(variant)]
show_table(pass_rate_table(all_rows, ["goal_preserved", "vague_not_frozen"]))
print("통과율은 실행을 하나하나 센 값이라 사례 사이의 난이도 차이가 섞여 있습니다. 판정은 위의 compare() 결과만 씁니다.")

### Langfuse에서 확인할 곳과 보고하는 법

Langfuse에서는 데이터셋 `day04_new-capability-v1`을 열면 변형마다 반복 세 개의 데이터셋 실행이 보입니다. 각 실행의 metadata에서 모델, effort, 두 sha256, 게이트 플래그, 임계값, `jev_model`을 확인하고, 실행 점수에서 검사별 통과율과 Wilson 구간, 확인 불가 수를 봅니다. 항목을 열면 그 실행의 추적(Claude 호출과 도구 호출)이 연결되어 있습니다. 이 화면의 구성은 설치된 SDK의 코드로 확인한 데까지만 적었고, 화면 자체는 이 노트북을 쓸 때 확인하지 않았습니다.

보고는 아래 틀을 그대로 채웁니다. 빈칸을 남기지 않고, 모르는 것은 "모른다"고 적습니다.

| 항목 | 적을 내용 |
|---|---|
| 측정 | dev 4 사례 x 3 반복 x 3 변형, 변형 이름, 실행 날짜 |
| 목표 지표별 판정 | 판정, `decide()`의 이유 문장 |
| 모르는 것 | 값 없는 가드 목록, 변형별 확인 불가 수, 오류로 끝난 실행과 원인 |
| 게이트 동작 | H2가 막은 횟수, H1이 사용자에게 물은 횟수, `decision_unavailable` 횟수 |
| 주의사항 | 두 게이트를 함께 켬, 임계값 0.5 미조정, dev 사례만 사용, 수업 사례는 S2/S3에 맞춰 고름 |

## 직접 해보기

이제 개선을 시도합니다. 규칙은 같습니다.

- **dev에서만 합니다.** holdout은 열지 않습니다.
- **바꾸기 전에 적습니다.** 무엇을 왜 바꾸고 어떤 판정을 예상하는지 카드로 먼저 남깁니다(해보기 1).
- **새 이름으로 돌립니다.** 실행기는 같은 이름을 거부합니다.
- **같은 규칙으로 비교합니다.** 수업의 A, A2와 비교하고 가드는 `prereg_guards`를 그대로 씁니다.
- 반복이 3보다 적은 결과는 **1회 관찰**이라 부르고 판정에 쓰지 않습니다.
- 바꿀 수 있는 것은 둘입니다. **임계값**(해보기 2)과 **H2의 Jev 질문 문구**(해보기 3)입니다. H1의 질문은 서버 자식 프로세스에 있어 노트북에서 바꿀 수 없습니다. 바꾸려면 `cap_lab/server/gate.py:28`을 직접 고쳐야 하고, 그러면 `server_sha256`이 바뀌어 metadata에 그 사실이 남습니다.

유료 셀은 `RUN_EXERCISE = False`로 잠겨 있습니다. 하나를 켜면 12회 실행, Claude 약 60~108회, Jev 최대 72회입니다.

### 해보기 1. 바꾸기 전에 내 사전 등록 카드를 쓴다

결과를 본 뒤에는 무엇을 예상했는지 정직하게 기억하기 어렵습니다. 아래 셀은 바꿀 내용, 목표 지표, 가드, 예상 판정을 JSON 카드로 쓰고 sha256을 출력합니다. 같은 이름의 카드가 이미 있으면 덮어쓰지 않고 멈춥니다. `why`와 `my_prediction`을 본인의 문장으로 바꾼 뒤 실행합니다.

In [ ]:
# [내 것 만들기] 바꾸기 전에 내 사전 등록 카드를 쓴다 (호출 0회)
MY_VARIANT_NAME = f"{RUN_PREFIX}-B-h2t07"   # 새 이름. 이미 쓴 이름은 실행기가 거부한다.
MY_CARD = {
    "variant": MY_VARIANT_NAME,
    "change": "H2 임계값 0.5 -> 0.7 (질문 문구와 H1은 그대로)",
    "why": "여기에 바꾸는 이유를 한 문장으로 적으세요.",
    "target": "goal_preserved",
    "guards": prereg_guards("goal_preserved"),
    "cases": list(CLASS_CASE_IDS),
    "repeats": CLASS_REPEATS,
    "compare_with": [VARIANT_A.name, VARIANT_A2.name],
    "decision_rule": "P5 그대로 (cap_lab/stats.py의 decide)",
    "my_prediction": "여기에 예상 판정을 적으세요(다섯 판정 가운데 하나와 그 이유).",
}

card_path = ROOT / "outputs" / "class" / f"card-{MY_VARIANT_NAME}.json"
if card_path.exists():
    raise RuntimeError(f"{card_path.name}이 이미 있습니다. 결과를 본 뒤 카드를 고쳐 쓰지 않습니다. 새 이름으로 새 카드를 쓰세요.")
card_path.parent.mkdir(parents=True, exist_ok=True)
card_text = json.dumps(MY_CARD, ensure_ascii=False, indent=2)
card_path.write_text(card_text, encoding="utf-8")
print("카드를 썼습니다:", card_path.relative_to(ROOT))
print("카드 sha256:", hashlib.sha256(card_text.encode("utf-8")).hexdigest())

### 해보기 2. 임계값을 바꿔 새 이름으로 다시 측정한다

H2 임계값을 0.7로 올려 봅니다. 높이면 GoalGuard가 더 자주 막습니다. 목표 지표가 오를 수도 있지만, 모델이 다시 부르느라 `model_calls`와 `latency_s`가 늘어 `교환`이 될 수도 있습니다. 이 0.7은 연습용 값이며 사전 등록된 임계값이 아닙니다. 임계값을 정식으로 정하는 일은 저자가 검토한 dev 라벨로 합니다(P4).

임계값은 실행 metadata의 `threshold_h2`에 그대로 기록되므로 따로 적을 필요가 없습니다. H1 임계값을 바꾸려면 `threshold_h1`을 바꿉니다. 그 값은 서버의 `--gate-threshold` 인자로 넘어갑니다(`cap_lab/experiment.py:200`).

In [ ]:
# [바꿔보기] 임계값을 바꾼 새 변형을 dev 4 사례로 다시 측정한다 (RUN_EXERCISE로 잠김)
RUN_EXERCISE = False

VARIANT_MY_THRESHOLD = Variant(
    MY_VARIANT_NAME,
    gate_h1=True,
    gate_h2=True,
    threshold_h1=VARIANT_B.threshold_h1,   # H1은 그대로 두고
    threshold_h2=0.7,                       # H2만 바꾼다. 연습용 값이며 사전 등록된 임계값이 아니다.
    metadata={"measurement": "class-exercise", "card": f"card-{MY_VARIANT_NAME}.json"},
)
if RUN_EXERCISE:
    stop_unless_langfuse_ready()
    announce(f"{VARIANT_MY_THRESHOLD.name} 실행: 사례 4 x 반복 3", model_calls=108, jev_calls=72, runs=12)
    names = await run_experiment(VARIANT_MY_THRESHOLD, CLASS_CASES, CLASS_REPEATS, settings=SETTINGS,
                                 mirror_dir=CLASS_MIRROR_DIR)
    print("실행 이름:", names)
else:
    print("RUN_EXERCISE가 False라 실행하지 않았습니다. 켜면 실행 12회, Claude 약 60~108회, Jev 최대 72회입니다.")

### 해보기 3. H2 질문 문구를 바꿔 새 이름으로 다시 측정한다

H2의 질문은 `cap_lab/middleware.py:25`의 `GOAL_PRESERVED`입니다. GoalGuard는 판정할 때마다 이 모듈 변수를 읽으므로(`cap_lab/middleware.py:106`), 측정하는 동안만 다른 질문으로 바꿔 두면 B의 H2가 새 질문으로 묻습니다. H2는 이 노트북 프로세스에서 돌기 때문에 가능한 방법입니다. 아래 `goal_question`은 블록이 끝나면 원래 질문으로 되돌립니다.

질문 문구는 실행 metadata에 자동으로 기록되지 않습니다(4절). 그래서 질문의 sha256을 변형의 `metadata`에 직접 넣습니다. 문구 전체를 넣지 않는 이유는 Langfuse SDK가 200자를 넘는 전파 속성 값을 버리기 때문입니다(`.venv/lib/python3.13/site-packages/langfuse/_client/propagation.py:632`). 문구 전체는 해보기 1의 카드에 적어 둡니다.

질문의 `name`은 `goal_preserved`로 둡니다. GoalGuard가 답을 이 이름으로 꺼내기 때문입니다.

In [ ]:
# [바꿔보기] H2 질문 문구를 바꾼 새 변형을 측정한다 (RUN_EXERCISE로 잠김)
from cap_lab import middleware

MY_GOAL_QUESTION = Noul(
    name="goal_preserved",   # 이름은 그대로 둔다. GoalGuard가 이 이름으로 답을 꺼낸다(cap_lab/middleware.py:106).
    instructions=GOAL_PRESERVED.instructions + " 조사나 띄어쓰기만 다르면 같은 목표로 본다.",   # 여기를 고친다
    true=GOAL_PRESERVED.true,
    false=GOAL_PRESERVED.false,
)
question_sha256 = hashlib.sha256(
    json.dumps(asdict(MY_GOAL_QUESTION), ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()


@contextmanager
def goal_question(question: Noul):
    '''GoalGuard가 판정할 때 읽는 cap_lab.middleware.GOAL_PRESERVED를 블록 안에서만 바꾼다. 끝나면 되돌린다.

    H2는 이 노트북 프로세스에서 돌기 때문에 바뀐다. H1의 TESTABILITY는 서버 자식 프로세스에 있어 이 방법으로는 바뀌지 않는다.
    '''
    original = middleware.GOAL_PRESERVED
    middleware.GOAL_PRESERVED = question
    try:
        yield
    finally:
        middleware.GOAL_PRESERVED = original


RUN_EXERCISE = False
VARIANT_MY_QUESTION = Variant(
    f"{RUN_PREFIX}-B-h2q1",
    gate_h1=True,
    gate_h2=True,
    metadata={"measurement": "class-exercise", "h2_question_sha256": question_sha256},
)
print("내 질문 sha256:", question_sha256)
if RUN_EXERCISE:
    stop_unless_langfuse_ready()
    announce(f"{VARIANT_MY_QUESTION.name} 실행: 사례 4 x 반복 3", model_calls=108, jev_calls=72, runs=12)
    with goal_question(MY_GOAL_QUESTION):
        names = await run_experiment(VARIANT_MY_QUESTION, CLASS_CASES, CLASS_REPEATS, settings=SETTINGS,
                                     mirror_dir=CLASS_MIRROR_DIR)
    print("실행 이름:", names)
    print("원래 질문으로 되돌렸는가:", middleware.GOAL_PRESERVED == GOAL_PRESERVED)
else:
    print("RUN_EXERCISE가 False라 실행하지 않았습니다. 켜면 실행 12회, Claude 약 60~108회, Jev 최대 72회입니다.")

### 해보기 4. 새 변형을 A, A2와 비교하고 카드와 맞춰 본다

새 변형을 수업의 A, A2와 같은 규칙으로 비교합니다. 잡음 폭은 이미 잰 A 대 A2에서 나오므로 새로 돌릴 필요가 없습니다. 반복 3회가 모두 있을 때만 비교하고, 그렇지 않으면 1회 관찰로만 남깁니다. 판정이 나오면 해보기 1의 카드에 적은 예측과 나란히 적습니다.

In [ ]:
# [바꿔보기] 새 변형을 수업의 A, A2와 비교하고 카드의 예측과 맞춰 본다 (호출 0회)
COMPARE_NAME = MY_VARIANT_NAME          # 해보기 3의 변형을 보려면 VARIANT_MY_QUESTION.name으로 바꾼다
COMPARE_TARGET = "goal_preserved"       # H1 임계값을 바꿨다면 "vague_not_frozen"

new_files = mirror_files(COMPARE_NAME)
if len(new_files) != CLASS_REPEATS:
    print(f"{COMPARE_NAME}의 반복 {CLASS_REPEATS}회 기록이 다 있지 않습니다(지금 {len(new_files)}개). "
          "해보기 2나 3을 먼저 끝내세요. 3회보다 적은 기록은 1회 관찰로만 읽습니다.")
else:
    exercise_result = compare(
        mirror_files(VARIANT_A.name), mirror_files(VARIANT_A2.name), new_files,
        target=COMPARE_TARGET, guards=prereg_guards(COMPARE_TARGET), split="dev",
    )
    report(exercise_result)
    print("카드의 예측:", MY_CARD["my_prediction"])

**관찰할 것.**

- 판정은 수업 측정과 같은 이유로 `회귀`, `개선 확인 불가`, `판정 불가: 데이터 부족` 셋 가운데 하나일 것입니다. 같은 4 사례를 썼기 때문입니다.
- dev에서 좋아 보이는 결과가 나와도 그것은 dev를 보며 고친 결과입니다. 고친 사람이 이미 그 사례를 알고 있으므로 낙관적으로 기웁니다. 최종 주장은 저자가 holdout을 한 번 열어서만 합니다.
- 판정이 마음에 들지 않을 때 문구를 조금 고쳐 또 돌리고, 또 돌리는 일을 반복하면, 여러 번 시도한 끝에 우연히 좋게 나온 하나를 고르게 됩니다. 시도마다 카드를 쓰고, 좋지 않은 시도도 기록에서 지우지 않습니다. 몇 번 시도했는지가 결과의 일부입니다.

## 확인 질문

1. 한 실행에서 S3와 S5가 함께 실패했습니다. `attribute()`는 이 실행을 어디에 세며, 왜 그렇게 정했습니까? 그 규칙이 실제 원인을 놓칠 수 있는 경우를 하나 드십시오.
2. P3의 세 조건 가운데 코드로 계산되는 것은 무엇입니까? 나머지 둘은 무엇으로 보입니까? H1의 자리에 대해 3번 조건을 반례로 설명해 보십시오.
3. A2를 돌리지 않으면 `compare()`와 `decide()`는 어느 단계에서 무엇을 냅니까? A2 대신 "잡음 폭은 0이라고 치자"고 하면 무엇이 잘못됩니까?
4. 수업의 4 사례로는 `개선`이 나올 수 없습니다. 1절의 어느 표, 어느 칸이 그 이유를 보여 줍니까? 이 한계를 없애려면 무엇이 필요합니까?
5. 오류로 끝난 실행을 실패로 세면 판정이 어떻게 틀어집니까? 오류 줄을 지워 버리면 무엇을 잃습니까?
6. Jev가 응답하지 못했을 때 GoalGuard와 H1 게이트는 각각 무엇을 합니까? 통과시키지 않는 이유를 측정의 관점에서 설명해 보십시오.
7. 실행기가 같은 변형 이름을 거부하는 이유는 무엇입니까? 측정이 중간에 끊겼을 때는 어떻게 다시 재야 합니까?
8. dev에서 임계값을 고쳐 좋은 판정을 얻었습니다. 이것을 최종 주장으로 쓸 수 없는 이유는 무엇이고, 무엇이 더 필요합니까?

## 정리

- **병목은 세어서 찾습니다.** 실행마다 첫 실패 단계 하나에 귀속하면 같은 원인을 두 번 세지 않습니다. 다만 판단 모델을 둘 자리는 수만으로 정하지 않습니다. 판단 형태의 문제인지, 결정적 규칙으로 풀 수 없는지를 반례로 따집니다(P3). H1(S3)과 H2(S2)는 그 논증으로 세운 가설이고, 기준선이 가설과 어긋나면 게이트를 옮기기 전에 저자에게 묻습니다.
- **게이트는 fail-closed입니다.** Jev가 답하지 못하면 H2는 도구를 부르지 않고, H1은 아무것도 기록하지 않습니다. 임계값 0.5는 라벨로 재기 전까지 자리표시자입니다.
- **개선은 미리 정한 규칙으로만 말합니다.** A/A2로 우연의 폭을 재고, 사례 단위 쌍체 부트스트랩으로 구간을 내고, `decide()`의 순서대로 판정합니다. 잡음 폭을 빼거나, None을 실패로 세거나, 가드를 고르면 같은 기록에서 다른 판정이 나옵니다. 5절에서 셋 다 직접 보았습니다.
- **기록은 Langfuse와 로컬 사본 두 곳에 남습니다.** 인증이 통과해야 측정이 시작되고, 실행마다 모델, sha256, 게이트, 임계값, Jev 모델이 고정해 기록됩니다. 판정은 오류 항목까지 담은 로컬 사본으로 냅니다.
- **'개선 확인 불가'와 '판정 불가: 데이터 부족'은 정당한 결과입니다.** 수업 측정의 4 사례는 가드 다섯 개에 값이 없어 `개선`에 이를 수 없습니다. 그 사실을 판정과 함께 그대로 보고하는 것이 이 실습의 결론입니다.